### DATA LOADING MODULE
- Imports

In [4]:

import os
import time
import json
import warnings
import requests
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

True

- CONFIGURATION 

In [5]:
DUNE_API_KEYS = [
    os.getenv("DUNE_LASEVEN7"),
    os.getenv("DUNE_FIRSTBML"),
    os.getenv("DUNE_LASEVEN71"),
    os.getenv("DUNE_LASEVEN7_TEAM"),
    os.getenv("DUNE_FIRSTBML_TEAM")
]

DUNE_API_KEYS = [key for key in DUNE_API_KEYS if key and str(key).strip()]
print("🔍 Loaded API Keys:")
env_names = [
    "DUNE_LASEVEN7", 
    "DUNE_FIRSTBML",
    "DUNE_LASEVEN71",
    "DUNE_LASEVEN7_TEAM",
    "DUNE_FIRSTBML_TEAM"
]
for i, key in enumerate(DUNE_API_KEYS):
    env_name = env_names[i] if i < len(env_names) else "UNKNOWN"
    print(f"   ✅ {i+1}. {env_name}: {key[:8]}...")

if not DUNE_API_KEYS:
    raise ValueError("No Dune API keys found!")

COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

for d in ['data', 'data/price_cache', 'logs']:
    os.makedirs(d, exist_ok=True)

QUERIES = {
    "whales": ("6395391", "data/dune_whales_cache.json", "data/whale_ml_ready.csv"),
    "market_intent": ("6385600", "data/dune_intent_cache.json", "data/market_intent_ml_ready.csv")
}

DUNE_START = pd.Timestamp('2017-10-16', tz='UTC')
COINGECKO_BTC_START = '01-01-2013'
COINGECKO_ETH_START = '01-08-2015'


🔍 Loaded API Keys:
   ✅ 1. DUNE_LASEVEN7: D1Smlg20...
   ✅ 2. DUNE_FIRSTBML: fHg07khx...
   ✅ 3. DUNE_LASEVEN71: hueVX9VB...
   ✅ 4. DUNE_LASEVEN7_TEAM: 7ND5DKw8...
   ✅ 5. DUNE_FIRSTBML_TEAM: p2UxSBFJ...


- KEY ROTATION

In [6]:

class DuneKeyRotator:
    def __init__(self, api_keys):
        self.api_keys = api_keys
        self.key_index = 0
        self.key_usage = {key: {"count": 0, "last_used": None, "errors": 0, "exhausted": False} for key in api_keys}
        self.total_requests = 0
    
    def get_next_key(self):
        if not self.api_keys:
            raise ValueError("No API keys available")
        
        available_keys = []
        for key in self.api_keys:
            usage = self.key_usage[key]
            
            if usage["exhausted"]:  # Skip exhausted keys
                continue
                
            if usage["errors"] >= 3:
                continue
                
            if usage["last_used"]:
                time_since_use = (datetime.now() - usage["last_used"]).total_seconds()
                if time_since_use < 60:
                    continue
                    
            available_keys.append((key, usage["count"], usage["errors"]))
        
        if not available_keys:
            # Check if ALL keys are exhausted
            if all(usage["exhausted"] for usage in self.key_usage.values()):
                raise Exception("ALL_API_KEYS_EXHAUSTED")
            
            available_keys = [(key, self.key_usage[key]["count"], self.key_usage[key]["errors"]) 
                            for key in self.api_keys if not self.key_usage[key]["exhausted"]]
        
        if not available_keys:
            raise Exception("NO_KEYS_AVAILABLE")
        
        available_keys.sort(key=lambda x: (x[2], x[1]))
        selected_key = available_keys[0][0]
        
        self.key_usage[selected_key]["count"] += 1
        self.key_usage[selected_key]["last_used"] = datetime.now()
        self.total_requests += 1
        
        return selected_key
    
    def mark_exhausted(self, key):
        """Mark a key as exhausted (reached monthly limit)"""
        if key in self.key_usage:
            self.key_usage[key]["exhausted"] = True
            print(f"⛔ KEY {key[:8]}... MARKED AS EXHAUSTED (monthly limit reached)")
    
    def are_all_keys_exhausted(self):
        """Check if all keys are exhausted"""
        return all(usage["exhausted"] for usage in self.key_usage.values())
        
    def mark_error(self, key):
        if key in self.key_usage:
            self.key_usage[key]["errors"] += 1
    
    def mark_success(self, key):
        if key in self.key_usage and self.key_usage[key]["errors"] > 0:
            self.key_usage[key]["errors"] = max(0, self.key_usage[key]["errors"] - 1)
    
    def get_stats(self):
        return {
            "total_requests": self.total_requests,
            "active_keys": sum(1 for key, stats in self.key_usage.items() if stats["errors"] < 5)
        }

key_rotator = DuneKeyRotator(DUNE_API_KEYS)

- API CLIENT

In [7]:

class DuneAPIClient:
    def __init__(self, key_rotator):
        self.key_rotator = key_rotator
        self.session = requests.Session()
        self.session.headers.update({
            "Content-Type": "application/json",
            "Accept": "application/json"
        })
    
    def make_request(self, method, url, **kwargs):
        max_retries = len(self.key_rotator.api_keys)  # Try all keys
        
        for attempt in range(max_retries):
            try:
                api_key = self.key_rotator.get_next_key()
                headers = kwargs.get('headers', {}).copy()
                headers["x-dune-api-key"] = api_key
                kwargs['headers'] = headers
                
                response = self.session.request(method, url, **kwargs)
                
                if response.status_code == 402:
                    # This key is exhausted for the month
                    self.key_rotator.mark_exhausted(api_key)
                    self.key_rotator.mark_error(api_key)
                    
                    # Check if all keys are now exhausted
                    if self.key_rotator.are_all_keys_exhausted():
                        return response  # Return 402 to signal complete exhaustion
                    
                    continue  # Try another key
                    
                elif response.status_code == 200:
                    self.key_rotator.mark_success(api_key)
                    return response
                    
                else:
                    self.key_rotator.mark_error(api_key)
                    
                    if attempt == max_retries - 1:
                        return response
                        
                    time.sleep(2)
                    
            except Exception as e:
                self.key_rotator.mark_error(api_key)
                
                if attempt == max_retries - 1:
                    raise
        
        raise Exception(f"Failed after {max_retries} attempts")
    
    def get(self, url, **kwargs):
        return self.make_request("GET", url, **kwargs)
    
    def post(self, url, **kwargs):
        return self.make_request("POST", url, **kwargs)

dune_client = DuneAPIClient(key_rotator)



- UTILITY FUNCTIONS
 - Dune

In [8]:

def to_utc(ts):
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")


def fetch_dune_incremental(qid, cache_path, query_name="whale_data", force_fetch=False):
    """
    Stops on rate limit instead of skipping dates
    """
    today = pd.Timestamp.now(tz='UTC').normalize()
    yesterday = today - pd.Timedelta(days=1)
    
    print(f"\n📊 Fetching {query_name}...")
    print(f"   Using {len(DUNE_API_KEYS)} API keys")
    
    df_cached = pd.DataFrame()
    last_date = None
    
    if os.path.exists(cache_path) and not force_fetch:
        try:
            with open(cache_path) as f:
                cached = json.load(f)
            
            if 'data' in cached and cached['data']:
                df_cached = pd.DataFrame(cached["data"])
                
                if 'block_date' in df_cached.columns:
                    df_cached["block_date"] = pd.to_datetime(df_cached["block_date"], utc=True)
                    
                    if 'is_estimate' in df_cached.columns:
                        df_cached = df_cached.drop('is_estimate', axis=1)
                    
                    last_date = df_cached["block_date"].max()
                    
                    if last_date >= yesterday:
                        print(f"✅ {query_name} cache current ({last_date.date()})")
                        return df_cached
                    
                    print(f"📅 Cache: {last_date.date()}, fetching new data...")
                else:
                    last_date = DUNE_START
            else:
                last_date = DUNE_START
                
        except Exception as e:
            print(f"⚠️  Cache error: {e}")
            last_date = DUNE_START
    else:
        print(f"📝 Fetching from {DUNE_START.date()}...")
        last_date = DUNE_START
    
    fetch_start = last_date + pd.Timedelta(days=1) if last_date else DUNE_START
    fetch_end = yesterday
    
    if fetch_start > fetch_end:
        return df_cached
    
    print(f"🔍 Fetching: {fetch_start.date()} to {fetch_end.date()}")
    
    # FETCH ONE DAY AT A TIME
    all_new_rows = []
    current_date = fetch_start
    
    while current_date <= fetch_end:
        query_params = {
            "start_date": current_date.strftime("%Y-%m-%d"),
            "end_date": current_date.strftime("%Y-%m-%d")
        }
        
        try:
            execute_url = f"https://api.dune.com/api/v1/query/{qid}/execute"
            execute_payload = {"query_parameters": query_params}
            
            print(f"   🔸 {current_date.date()}...", end="")
            
            resp = dune_client.post(execute_url, json=execute_payload, timeout=60)
            
            # ✅ Stop on 402, don't skip
            if resp.status_code != 200:
                if resp.status_code == 402:
                    print(f" ❌ 402 RATE LIMIT")
                    print(f"\n⚠️  STOPPED at {current_date.date()} - rate limit hit")
                    print(f"   Cache saved up to: {last_date.date() if last_date else 'N/A'}")
                    print(f"   Resume tomorrow when limits reset")
                    break  # STOP completely, don't skip
                
                print(f" ❌ {resp.status_code}")
                current_date += pd.Timedelta(days=1)
                time.sleep(2)
                continue
            
            resp_json = resp.json()
            
            if 'execution_id' not in resp_json:
                print(f" ❌ No execution_id")
                current_date += pd.Timedelta(days=1)
                time.sleep(2)
                continue
            
            eid = resp_json["execution_id"]
            
            # Wait for completion
            for attempt in range(60):
                status_url = f"https://api.dune.com/api/v1/execution/{eid}/status"
                
                try:
                    status_resp = dune_client.get(status_url, timeout=30)
                    
                    if status_resp.status_code == 200:
                        status_data = status_resp.json()
                        state = status_data.get("state", "UNKNOWN")
                        
                        if state == "QUERY_STATE_COMPLETED":
                            break
                        elif state in ["QUERY_STATE_FAILED", "QUERY_STATE_CANCELLED"]:
                            print(f" ❌ {state}")
                            break
                    
                except Exception as e:
                    print(f" ⚠️ {e}")
                    break
                
                time.sleep(5)
            else:
                print(f" ⏱️ Timeout")
                current_date += pd.Timedelta(days=1)
                time.sleep(2)
                continue
            
            # Get results
            results_url = f"https://api.dune.com/api/v1/execution/{eid}/results"
            results_resp = dune_client.get(results_url, timeout=30)
            
            if results_resp.status_code == 200:
                results_data = results_resp.json()
                
                if 'result' in results_data and 'rows' in results_data['result']:
                    rows = results_data["result"]["rows"]
                    if rows:
                        all_new_rows.extend(rows)
                        print(f" ✅ {len(rows)} rows")
                    else:
                        print(f" ⚠️ No data")
                else:
                    print(f" ❌ No results")
            else:
                print(f" ❌ {results_resp.status_code}")
            
        except Exception as e:
            print(f" ❌ {str(e)[:50]}")
        
        current_date += pd.Timedelta(days=1)
        time.sleep(2)
    
    if not all_new_rows:
        print(f"⚠️  No new data fetched")
        return df_cached
    
    df_new = pd.DataFrame(all_new_rows)
    print(f"📥 Total: {len(df_new)} new rows")
    
    if 'block_date' not in df_new.columns:
        print(f"❌ Missing block_date!")
        return df_cached
    
    df_new["block_date"] = pd.to_datetime(df_new["block_date"], utc=True)
    
    # Merge with cache
    if not df_cached.empty:
        common_cols = list(set(df_cached.columns) & set(df_new.columns))
        df_cached = df_cached[common_cols]
        df_new = df_new[common_cols]
        
        df_combined = pd.concat([df_cached, df_new], ignore_index=True)
        df_combined = df_combined.drop_duplicates(
            subset=['block_date'],
            keep='last'
        ).sort_values('block_date').reset_index(drop=True)
        
        print(f"📊 Combined: {len(df_combined)} rows")
    else:
        df_combined = df_new
        print(f"📊 New dataset: {len(df_combined)} rows")
    
    # Update cache
    update_cache_file(cache_path, df_combined)
    
    return df_combined
def update_cache_file(cache_path, df):
    try:
        if 'block_date' in df.columns:
            df_dates = df['block_date'].copy()
            
            df_serializable = df.copy()
            df_serializable['block_date'] = df_serializable['block_date'].dt.strftime('%Y-%m-%d')
            
            with open(cache_path, 'w') as f:
                json.dump({
                    "last_block_date": df_dates.max().strftime("%Y-%m-%d"),
                    "data": json.loads(df_serializable.to_json(orient="records", date_format='iso'))
                }, f, indent=2)
            
            print(f"💾 Cache updated to {df_dates.max().date()}")
            
    except Exception as e:
        print(f"❌ Cache update failed: {e}")


- Data Fetching for Bitcoin and Etherum from Coingecko

In [9]:

def fetch_cg_chunked(cg_id, start_date_str, end_date, key, days=30):
    url = "https://pro-api.coingecko.com/api/v3"
    headers = {"x-cg-pro-api-key": key}
    
    if isinstance(start_date_str, str):
        try:
            start_dt = pd.to_datetime(start_date_str, format='%d-%m-%Y', utc=True)
        except:
            start_dt = pd.to_datetime(start_date_str, utc=True)
    else:
        start_dt = to_utc(start_date_str)
    
    end_dt = to_utc(end_date) + pd.Timedelta(days=1)
    
    all_prices, curr = [], start_dt
    
    while curr < end_dt:
        next_dt = min(curr + pd.Timedelta(days=days), end_dt)
        params = {
            "vs_currency": "usd", 
            "from": int(curr.timestamp()), 
            "to": int(next_dt.timestamp())
        }
        
        try:
            r = requests.get(
                f"{url}/coins/{cg_id}/market_chart/range", 
                params=params, 
                headers=headers, 
                timeout=30
            )
            
            if r.status_code == 200:
                prices = r.json().get("prices", [])
                if prices:
                    all_prices.extend(prices)
            
        except Exception:
            pass
        
        time.sleep(0.5)
        curr = next_dt
    
    if not all_prices:
        return pd.DataFrame()
    
    df = pd.DataFrame(all_prices, columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("D")
    return df.groupby("date")["price"].mean().reset_index()

In [10]:
def get_price_incremental(symbol, cg_id, start_date_str, end, key, force_fetch=False):
    cache_path = f"data/price_cache/{symbol}.csv"
    today_utc = pd.Timestamp.utcnow().floor("D")
    yesterday = today_utc - pd.Timedelta(days=1)
    end_dt = min(to_utc(end), yesterday)
    
    print(f"\n💰 Fetching {symbol.upper()} prices...")
    
    if force_fetch and os.path.exists(cache_path):
        os.remove(cache_path)
    
    df_cached = pd.DataFrame()
    if os.path.exists(cache_path) and not force_fetch:
        try:
            df_cached = pd.read_csv(cache_path, parse_dates=["date"])
            df_cached["date"] = df_cached["date"].apply(to_utc)
            
            if not df_cached.empty:
                last_date = df_cached["date"].max()
                first_date = df_cached["date"].min()
                expected_start = to_utc(start_date_str)
                
                needs_historical = first_date > expected_start
                needs_updates = last_date < end_dt
                
                if not needs_historical and not needs_updates:
                    print(f"✅ {symbol.upper()} current ({first_date.date()} to {last_date.date()})")
                    return df_cached
                
                fetch_ranges = []
                
                if needs_historical:
                    fetch_ranges.append((expected_start, first_date - pd.Timedelta(days=1)))
                
                if needs_updates:
                    fetch_ranges.append((last_date + pd.Timedelta(days=1), end_dt))
                
                all_new_data = []
                for fetch_start, fetch_end in fetch_ranges:
                    if fetch_start <= fetch_end:
                        new_data = fetch_cg_chunked(cg_id, fetch_start, fetch_end, key)
                        if not new_data.empty:
                            all_new_data.append(new_data)
                
                if not all_new_data:
                    return df_cached
                
                df_new = pd.concat(all_new_data, ignore_index=True)
                df_new = df_new.rename(columns={"price": f"{symbol}_price"})
                
                df_combined = pd.concat([df_cached, df_new], ignore_index=True)
                df_combined = df_combined.drop_duplicates("date").sort_values("date").reset_index(drop=True)
                
                print(f"📊 {symbol.upper()}: {len(df_combined)} total")
                
        except Exception as e:
            print(f"⚠️  Cache error: {e}")
            df_cached = pd.DataFrame()
            fetch_start = to_utc(start_date_str)
    else:
        fetch_start = to_utc(start_date_str)
    
    if df_cached.empty:
        new_data = fetch_cg_chunked(cg_id, fetch_start, end_dt, key)
        
        if new_data.empty:
            return pd.DataFrame()
        
        df_combined = new_data.rename(columns={"price": f"{symbol}_price"})
    
    df_combined.to_csv(cache_path, index=False)
    print(f"💾 {symbol.upper()} saved: {df_combined['date'].min().date()} to {df_combined['date'].max().date()}")
    
    return df_combined


In [11]:
# ========== ADD TO DATA LOADER (after other imports) ==========
# BINANCE FUNDING RATE CONFIGURATION
CROWDED_LONG = 0.03    # +3% per 8h (very aggressive longs)
CROWDED_SHORT = -0.02  # -2% per 8h (short squeeze risk)

def fetch_binance_funding_incremental(symbol="ETHUSDT", start_date=None, end_date=None, force_fetch=False):
    """
    Fetch Binance funding rate data incrementally with caching
    """
    cache_path = "data/funding_rates_cache.json"
    cache_file = "data/funding_rates.csv"
    
    print(f"\n💰 Fetching Binance {symbol} funding rates...")
    
    # Load cached data if exists
    df_cached = pd.DataFrame()
    last_date = None
    
    if os.path.exists(cache_file) and not force_fetch:
        try:
            df_cached = pd.read_csv(cache_file, parse_dates=["funding_time"])
            df_cached["funding_time"] = df_cached["funding_time"].apply(to_utc)
            
            if not df_cached.empty:
                last_date = df_cached["funding_time"].max()
                print(f"📅 Cache: {len(df_cached)} records up to {last_date.date()}")
        except Exception as e:
            print(f"⚠️ Funding cache error: {e}")
            df_cached = pd.DataFrame()
    
    # Determine date range to fetch
    if start_date is None:
        if last_date:
            fetch_start = last_date + pd.Timedelta(hours=8)  # Next 8h slot
        else:
            fetch_start = pd.Timestamp("2020-01-01", tz="UTC")
    else:
        fetch_start = to_utc(start_date)
    
    if end_date is None:
        end_date = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=8)  # Avoid partial data
    else:
        end_date = to_utc(end_date)
    
    if fetch_start >= end_date:
        print(f"✅ Funding data current")
        return df_cached
    
    print(f"🔍 Fetching: {fetch_start.date()} to {end_date.date()}")
    
    # Binance API returns up to 1000 records per call
    # Each record is 8h, so 1000 records ≈ 333 days
    all_new_data = []
    
    try:
        url = "https://fapi.binance.com/fapi/v1/fundingRate"
        params = {
            "symbol": symbol,
            "limit": 1000
        }
        
        response = requests.get(url, params=params, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            
            if data:
                df_new = pd.DataFrame(data)
                df_new['funding_time'] = pd.to_datetime(df_new['fundingTime'], unit='ms', utc=True)
                df_new['funding_rate_8h'] = df_new['fundingRate'].astype(float)
                df_new = df_new[['funding_time', 'funding_rate_8h']]
                
                # Filter to requested date range
                df_new = df_new[(df_new['funding_time'] >= fetch_start) & 
                               (df_new['funding_time'] <= end_date)]
                
                if not df_new.empty:
                    all_new_data.append(df_new)
                    print(f"📥 Fetched {len(df_new)} funding records")
                else:
                    print(f"⚠️ No new funding data in range")
            else:
                print(f"⚠️ Empty response from Binance")
        else:
            print(f"⚠️ Binance API error: {response.status_code}")
            
    except Exception as e:
        print(f"❌ Funding fetch error: {e}")
    
    if not all_new_data:
        print(f"⚠️ No new funding data fetched")
        return df_cached
    
    # Combine new data with cache
    df_new_combined = pd.concat(all_new_data, ignore_index=True)
    
    if not df_cached.empty:
        # Remove overlapping dates from cache
        df_cached = df_cached[df_cached['funding_time'] < df_new_combined['funding_time'].min()]
        df_combined = pd.concat([df_cached, df_new_combined], ignore_index=True)
    else:
        df_combined = df_new_combined
    
    # Sort and deduplicate
    df_combined = df_combined.sort_values('funding_time').drop_duplicates('funding_time')
    
    # Save to cache
    df_combined.to_csv(cache_file, index=False)
    print(f"💾 Funding data saved: {len(df_combined)} records")
    print(f"   Date range: {df_combined['funding_time'].min().date()} to {df_combined['funding_time'].max().date()}")
    
    return df_combined

def process_funding_for_ml(df_funding, df_whales):
    """
    Convert 8h funding rates to daily and align with whale data dates
    """
    if df_funding.empty:
        print("⚠️ No funding data available, using zeros")
        return pd.DataFrame(columns=['block_date', 'eth_funding_rate_8h'])
    
    # Convert to daily (mean of 3 funding periods per day)
    df_funding['date'] = df_funding['funding_time'].dt.date
    daily_funding = (
        df_funding
        .groupby('date')['funding_rate_8h']
        .mean()
        .reset_index()
        .rename(columns={'funding_rate_8h': 'eth_funding_rate_8h'})
    )
    
    # Get whale data date range
    whale_dates = pd.to_datetime(df_whales['block_date']).dt.date
    min_date = whale_dates.min()
    max_date = whale_dates.max()
    
    # Create full date range
    all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
    all_dates_df = pd.DataFrame({'date': all_dates.date})
    
    # Merge funding data
    df_merged = pd.merge(
        all_dates_df,
        daily_funding,
        on='date',
        how='left'
    )
    
    # Fill missing dates (forward fill, then backward fill, then 0)
    df_merged['eth_funding_rate_8h'] = (
        df_merged['eth_funding_rate_8h']
        .ffill()
        .bfill()
        .fillna(0)
    )
    
    # Add block_date for merging
    df_merged['block_date'] = pd.to_datetime(df_merged['date'])
    
    return df_merged[['block_date', 'eth_funding_rate_8h']]

def load_all_data_incremental(force_fetch=False):
    """
    Load all data including funding rates
    """
    print("📊 Loading data...")
    
    # Load Dune and CoinGecko data first
    df_whales = fetch_dune_incremental(
        QUERIES["whales"][0], 
        QUERIES["whales"][1],
        query_name="whale_data",
        force_fetch=force_fetch
    )
    df_whales.to_csv(QUERIES["whales"][2], index=False)
    
    time.sleep(2)
    
    df_market = fetch_dune_incremental(
        QUERIES["market_intent"][0], 
        QUERIES["market_intent"][1],
        query_name="market_intent",
        force_fetch=force_fetch
    )
    df_market.to_csv(QUERIES["market_intent"][2], index=False)
    
    # Get max date for price fetching
    max_date = max(
        df_whales["block_date"].max() if not df_whales.empty else DUNE_START,
        df_market["block_date"].max() if not df_market.empty else DUNE_START
    )
    
    # Fetch prices
    df_btc = get_price_incremental("btc", "bitcoin", COINGECKO_BTC_START, max_date, COINGECKO_API_KEY, force_fetch)
    df_eth = get_price_incremental("eth", "ethereum", COINGECKO_ETH_START, max_date, COINGECKO_API_KEY, force_fetch)
    
    # ========== FETCH FUNDING DATA ==========
    # Get date range from whale data for funding fetch
    if not df_whales.empty:
        funding_start = df_whales['block_date'].min() - pd.Timedelta(days=7)  # Buffer
        funding_end = df_whales['block_date'].max()
        
        df_funding = fetch_binance_funding_incremental(
            symbol="ETHUSDT",
            start_date=funding_start,
            end_date=funding_end,
            force_fetch=force_fetch
        )
        
        # Process funding for ML
        df_funding_processed = process_funding_for_ml(df_funding, df_whales)
        
        # Save funding data
        funding_file = "data/funding_rates_ml_ready.csv"
        df_funding_processed.to_csv(funding_file, index=False)
        print(f"💾 Funding data saved: {funding_file}")
        
        # Print funding statistics
        if not df_funding_processed.empty:
            funding_stats = df_funding_processed['eth_funding_rate_8h']
            print(f"📊 Funding statistics:")
            print(f"   Mean: {funding_stats.mean():.6f}")
            print(f"   Min: {funding_stats.min():.6f}")
            print(f"   Max: {funding_stats.max():.6f}")
            print(f"   > {CROWDED_LONG}: {(funding_stats > CROWDED_LONG).sum()} days")
            print(f"   < {CROWDED_SHORT}: {(funding_stats < CROWDED_SHORT).sum()} days")
    
    print(f"\n📈 Summary:")
    print(f"   Whale: {len(df_whales)} rows")
    print(f"   Market: {len(df_market)} rows")
    print(f"   BTC: {len(df_btc)} rows")
    print(f"   ETH: {len(df_eth)} rows")
    print(f"   Funding: {len(df_funding_processed) if 'df_funding_processed' in locals() else 0} rows")
    
    return df_whales, df_market, df_btc, df_eth, df_funding_processed

def load_cached_data():
    """
    Load all cached data including funding rates
    """
    print("📂 Loading cached data...")
    
    files = {
        'whale': 'data/whale_ml_ready.csv',
        'market': 'data/market_intent_ml_ready.csv',
        'btc': 'data/price_cache/btc.csv',
        'eth': 'data/price_cache/eth.csv',
        'funding': 'data/funding_rates_ml_ready.csv'
    }
    
    loaded = {}
    
    for name, path in files.items():
        if os.path.exists(path):
            try:
                if name in ['btc', 'eth']:
                    df = pd.read_csv(path, parse_dates=["date"])
                    df["date"] = df["date"].apply(to_utc)
                elif name == 'funding':
                    df = pd.read_csv(path, parse_dates=["block_date"])
                    df["block_date"] = df["block_date"].apply(to_utc)
                else:
                    df = pd.read_csv(path, parse_dates=["block_date"])
                    df["block_date"] = df["block_date"].apply(to_utc)
                
                loaded[name] = df
                print(f"✅ {name}: {len(df)} rows")
            except Exception as e:
                print(f"❌ {name}: {e}")
                loaded[name] = pd.DataFrame()
        else:
            print(f"⚠️  {name} not found")
            loaded[name] = pd.DataFrame()
    
    return (loaded.get('whale', pd.DataFrame()),
            loaded.get('market', pd.DataFrame()),
            loaded.get('btc', pd.DataFrame()),
            loaded.get('eth', pd.DataFrame()),
            loaded.get('funding', pd.DataFrame()))

- Incrementally Loading and from dune and coingecko

- Main Loading Execution

In [14]:

if __name__ == "__main__":
    print("\n" + "="*70)
    print("📊 DATA LOADER - FIXED VERSION")
    print("="*70)
    
    print("\n📋 OPTIONS:")
    print("1️⃣  Fetch fresh (incremental)")
    print("2️⃣  Force re-fetch all")
    print("3️⃣  Load cached only")
    print("="*70)

    choice = input("\nSelect (1-3): ").strip()

    if choice == '1':
        print("\n🚀 Fetching fresh data...")
        load_all_data_incremental(force_fetch=False)
        
    elif choice == '2':
        confirm = input("Delete cache and fetch all? (y/n): ").lower()
        if confirm == 'y':
            load_all_data_incremental(force_fetch=True)
        
    elif choice == '3':
        load_cached_data()
        
    else:
        print("❌ Invalid option")
    
    print(f"\n{'='*70}")
    print("✅ Complete!")
    print(f"{'='*70}")


📊 DATA LOADER - FIXED VERSION

📋 OPTIONS:
1️⃣  Fetch fresh (incremental)
2️⃣  Force re-fetch all
3️⃣  Load cached only

🚀 Fetching fresh data...
📊 Loading data...

📊 Fetching whale_data...
   Using 5 API keys
📅 Cache: 2026-01-04, fetching new data...
🔍 Fetching: 2026-01-05 to 2026-01-06
   🔸 2026-01-05...⛔ KEY D1Smlg20... MARKED AS EXHAUSTED (monthly limit reached)
⛔ KEY fHg07khx... MARKED AS EXHAUSTED (monthly limit reached)
 ✅ 1 rows
   🔸 2026-01-06... ✅ 1 rows
📥 Total: 2 new rows
📊 Combined: 3005 rows
💾 Cache updated to 2026-01-06

📊 Fetching market_intent...
   Using 5 API keys
📅 Cache: 2026-01-04, fetching new data...
🔍 Fetching: 2026-01-05 to 2026-01-06
   🔸 2026-01-05... ✅ 1 rows
   🔸 2026-01-06... ✅ 1 rows
📥 Total: 2 new rows
📊 Combined: 3005 rows
💾 Cache updated to 2026-01-06

💰 Fetching BTC prices...
📊 BTC: 4635 total
💾 BTC saved: 2013-04-28 to 2026-01-06

💰 Fetching ETH prices...
📊 ETH: 3805 total
💾 ETH saved: 2015-08-07 to 2026-01-06

💰 Fetching Binance ETHUSDT funding rat

- Exploratory Data Analysis (EDA)

In [15]:
pd.read_csv('data/whale_ml_ready.csv')

,deposit_tx_count,net_flow_ma7,non_exchange_ratio,std_whale_tx_size_eth,deposit_withdrawal_ratio,exchange_volume_ratio,withdrawal_tx_count,whale_net_exchange_flow_eth,whale_tx_count,block_date,mega_whale_volume_eth,non_exchange_volume_eth,whale_exchange_withdrawals_eth,whale_volume_eth,mega_whale_tx_count,non_exchange_tx_count,whale_exchange_deposits_eth,mega_whale_ratio
0,9.0,10749.2728,0.9098,2310.607296,0.5146,0.0902,5.0,10749.2728,190.0,2017-10-16 00:00:00+00:00,3.335630e+05,3.382636e+05,22147.0794,3.718084e+05,149.0,176.0,11397.8066,0.8971
1,21.0,-20684.9471,0.9115,13716.203240,1.5295,0.0885,15.0,-20684.9471,250.0,2017-10-17 00:00:00+00:00,1.055398e+06,1.017314e+06,39067.9609,1.116134e+06,185.0,214.0,59752.9080,0.9456
2,5.0,-15211.3463,0.9586,3957.459236,1.6254,0.0414,5.0,-9737.7455,349.0,2017-10-18 00:00:00+00:00,9.872356e+05,9.463566e+05,15570.6242,9.872356e+05,349.0,339.0,25308.3697,1.0000
3,6.0,3912.9137,0.8966,2819.239036,0.1403,0.1034,6.0,42161.4337,336.0,2017-10-19 00:00:00+00:00,4.221430e+05,4.850454e+05,49041.2315,5.409664e+05,217.0,324.0,6879.7978,0.7803
4,11.0,3137.9499,0.9381,3236.014159,0.9605,0.0619,10.0,813.0587,319.0,2017-10-20 00:00:00+00:00,5.448516e+05,6.113884e+05,20589.5335,6.517544e+05,212.0,298.0,19776.4748,0.8360
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3000,18.0,-38461.2988,0.5902,4361.441154,1.6701,0.4098,7.0,-63049.3965,70.0,2026-01-02 00:00:00+00:00,6.130161e+05,3.617799e+05,94093.4141,6.130161e+05,70.0,45.0,157142.8106,1.0000
3001,8.0,-32946.0600,0.8439,12264.236700,3.1841,0.1561,2.0,-32946.0600,45.0,2026-01-03 00:00:00+00:00,4.043606e+05,3.412452e+05,15084.6473,4.043606e+05,45.0,35.0,48030.7073,1.0000
3002,11.0,-15832.0280,0.6605,3569.436301,1.6160,0.3395,2.0,-15832.0280,42.0,2026-01-04 00:00:00+00:00,1.980223e+05,1.307876e+05,25701.3466,1.980223e+05,42.0,29.0,41533.3746,1.0000
3003,16.0,-106383.3682,0.7245,6949.888079,3.0385,0.2755,4.0,-106383.3682,74.0,2026-01-05 00:00:00+00:00,7.650995e+05,5.543415e+05,52187.2732,7.650995e+05,74.0,54.0,158570.6414,1.0000


In [16]:
pd.read_csv('data/whale_ml_ready.csv').dtypes

deposit_tx_count                  float64
net_flow_ma7                      float64
non_exchange_ratio                float64
std_whale_tx_size_eth             float64
deposit_withdrawal_ratio          float64
exchange_volume_ratio             float64
withdrawal_tx_count               float64
whale_net_exchange_flow_eth       float64
whale_tx_count                    float64
block_date                         object
mega_whale_volume_eth             float64
non_exchange_volume_eth           float64
whale_exchange_withdrawals_eth    float64
whale_volume_eth                  float64
mega_whale_tx_count               float64
non_exchange_tx_count             float64
whale_exchange_deposits_eth       float64
mega_whale_ratio                  float64
dtype: object

In [17]:
pd.read_csv('data/whale_ml_ready.csv').isnull().sum()

deposit_tx_count                  0
net_flow_ma7                      0
non_exchange_ratio                0
std_whale_tx_size_eth             0
deposit_withdrawal_ratio          0
exchange_volume_ratio             0
withdrawal_tx_count               0
whale_net_exchange_flow_eth       0
whale_tx_count                    0
block_date                        0
mega_whale_volume_eth             0
non_exchange_volume_eth           0
whale_exchange_withdrawals_eth    0
whale_volume_eth                  0
mega_whale_tx_count               0
non_exchange_tx_count             0
whale_exchange_deposits_eth       0
mega_whale_ratio                  0
dtype: int64

In [18]:
pd.read_csv('data/market_intent_ml_ready.csv')

,whale_volume_ratio_delta_1d,net_exchange_flow_ratio,eth_burned_zscore_90d,tx_per_active_delta_1d,smart_contract_ratio_delta_1d,whale_tx_zscore_90d,whale_volume_ratio,whale_exchange_asymmetry,block_date,median_gas_delta_7d,tx_per_active_zscore_90d,whale_volume_ratio_delta_3d,exchange_flow_share,median_gas_delta_1d,whale_exchange_flow_ratio,eth_burned_delta_1d,block_fullness_delta_1d
0,NaN,0.002401,0.0000,NaN,NaN,0.0000,0.237421,0.274629,2017-10-16 00:00:00+00:00,NaN,0.0000,NaN,0.015021,NaN,0.001739,NaN,NaN
1,0.027758,-0.000963,0.0000,-0.0661,0.000083,0.7071,0.265178,-0.187957,2017-10-17 00:00:00+00:00,NaN,-0.7071,NaN,0.015033,-10.3854,-0.001727,NaN,-0.145119
2,-0.065376,-0.000241,0.0000,0.0456,0.024143,0.0901,0.199802,-0.250389,2017-10-18 00:00:00+00:00,NaN,0.2477,NaN,0.010296,3.2781,-0.000781,NaN,-0.057348
3,-0.005306,0.003385,0.0000,0.0392,0.043882,0.2059,0.194496,0.753946,2017-10-19 00:00:00+00:00,NaN,0.9788,-0.042924,0.008123,-0.1501,0.003410,NaN,0.058294
4,0.027682,0.001450,0.0000,0.0256,-0.022849,0.2428,0.222178,0.020142,2017-10-20 00:00:00+00:00,NaN,1.1727,-0.043000,0.007958,-3.0014,0.000072,NaN,0.062177
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3000,0.036573,-0.065930,-0.2337,0.0812,0.036647,0.8464,0.948359,-0.167513,2026-01-02 00:00:00+00:00,0.0586,-0.7325,0.004523,0.383195,0.0482,-0.063541,9.0262,-0.001872
3001,-0.030761,-0.028868,-0.2608,0.1529,-0.030336,-1.4696,0.917597,-0.124502,2026-01-03 00:00:00+00:00,0.0270,0.3186,-0.032726,0.211788,-0.0477,-0.025588,-7.5086,0.000748
3002,-0.012822,-0.014358,-0.2637,-0.2407,0.005039,-1.4965,0.904776,-0.051472,2026-01-04 00:00:00+00:00,0.0263,-1.3467,-0.007010,0.212475,-0.0015,-0.010591,-2.1134,0.000706
3003,0.044263,-0.074827,-0.2140,-0.0887,-0.002745,1.5301,0.949039,-0.226926,2026-01-05 00:00:00+00:00,0.0399,-1.9000,0.000680,0.324977,0.0747,-0.072996,12.5480,-0.001574


In [19]:
pd.read_csv('data/market_intent_ml_ready.csv').dtypes

whale_volume_ratio_delta_1d      float64
net_exchange_flow_ratio          float64
eth_burned_zscore_90d            float64
tx_per_active_delta_1d           float64
smart_contract_ratio_delta_1d    float64
whale_tx_zscore_90d              float64
whale_volume_ratio               float64
whale_exchange_asymmetry         float64
block_date                        object
median_gas_delta_7d              float64
tx_per_active_zscore_90d         float64
whale_volume_ratio_delta_3d      float64
exchange_flow_share              float64
median_gas_delta_1d              float64
whale_exchange_flow_ratio        float64
eth_burned_delta_1d              float64
block_fullness_delta_1d          float64
dtype: object

In [20]:
pd.read_csv('data/market_intent_ml_ready.csv').isnull().sum()

whale_volume_ratio_delta_1d         1
net_exchange_flow_ratio             0
eth_burned_zscore_90d               0
tx_per_active_delta_1d              1
smart_contract_ratio_delta_1d       1
whale_tx_zscore_90d                 0
whale_volume_ratio                  0
whale_exchange_asymmetry            0
block_date                          0
median_gas_delta_7d                 7
tx_per_active_zscore_90d            0
whale_volume_ratio_delta_3d         3
exchange_flow_share                 0
median_gas_delta_1d                 1
whale_exchange_flow_ratio           0
eth_burned_delta_1d              1390
block_fullness_delta_1d             1
dtype: int64

In [23]:
"""
ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT
FINAL HARDENING PHASE - ALL CRITICAL FIXES APPLIED
"""

import os
import time
import json
import warnings
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from dotenv import load_dotenv
import joblib

warnings.filterwarnings('ignore')
load_dotenv()

# Create directories
for d in ['validation', 'backtest', 'models']:
    os.makedirs(d, exist_ok=True)

# ========== FROZEN CONFIGURATION (DO NOT MODIFY) ==========
# Feature sets - FROZEN
SHORT_FEATURES = [
    'exchange_flow_share', 'net_exchange_flow_ratio', 'whale_exchange_flow_ratio',
    'whale_exchange_asymmetry', 'eth_vol7', 'eth_vol30', 'btc_ret_lag1',
    'btc_ret_lag3', 'eth_btc_corr_30d', 'whale_volume_ratio_delta_3d',
    'exchange_volume_zscore'
]

LONG_FEATURES = [
    'btc_rsi', 'eth_vol7', 'whale_volume_ratio', 'eth_rsi',
    'btc_ret_lag1', 'eth_burned_zscore_90d', 'eth_btc_corr_30d',
    'eth_ret_lag1', 'btc_ret_lag7', 'btc_vol30',
    'whale_volume_ratio_delta_1d', 'whale_volume_ratio_delta_3d',
    'exchange_flow_share', 'net_exchange_flow_ratio',
    'whale_exchange_flow_ratio', 'tx_per_active_zscore_90d'
]

# Trading parameters
SLIPPAGE = 0.0008
FEES = 0.0004

# Entry thresholds - FROZEN (can tune later)
LONG_ENTRY_THRESHOLD = 0.50  # Lower for LONG to allow confirmation rescue
SHORT_ENTRY_THRESHOLD = 0.55  # Higher for SHORT (requires stronger signal)

# ========== UNIFIED CONFIDENCE & POSITION SIZING ==========

def adjust_confidence_unified(prob, regime, direction=None, veto_score=0):
    """
    UNIFIED CONFIDENCE ADJUSTMENT
    Applies same saturation logic to both LONG and SHORT
    """
    # Base confidence from model
    base_conf = float(prob)
    
    # Apply veto boost (same for both directions)
    veto_boost = np.tanh(veto_score / 3) * 0.15
    
    # Apply confidence caps based on regime
    if regime == "R3":
        # R3: Early weakness - conservative
        max_conf = 0.70
    elif regime == "R5":
        # R5: Distribution - moderate confidence
        max_conf = 0.85
    elif regime in ["R1", "R2"]:
        # R1/R2: Bull regimes - conservative for LONG
        max_conf = 0.75
    else:
        max_conf = 0.95
    
    # Calculate adjusted confidence
    adj_conf = np.clip(base_conf + veto_boost, 0, max_conf)
    
    return adj_conf

def map_confidence_to_size_unified(conf, regime=None, direction=None):
    """
    UNIFIED POSITION SIZING with explicit regime-aware thresholds
    """
    # ✅ EXPLICIT asymmetric confidence floors
    if direction == "LONG":
        if conf < 0.50:  # Lower threshold for LONG
            return 0.0
    elif direction == "SHORT":
        if conf < 0.55:  # Higher threshold for SHORT
            return 0.0
    else:
        if conf < 0.55:  # Default
            return 0.0
    
    # Base sizing scale (same for both directions)
    if conf < 0.60: 
        base_size = 0.25
    elif conf < 0.65: 
        base_size = 0.50
    elif conf < 0.70: 
        base_size = 0.75
    elif conf < 0.75: 
        base_size = 1.00
    elif conf < 0.80: 
        base_size = 1.25
    else: 
        base_size = 1.50
    
    # ✅ Apply regime-specific caps (explicit)
    if regime in ["R3", "R5"]:
        # SHORT regimes: conservative
        base_size = min(base_size, 1.0)
    elif regime in ["R1", "R2"]:
        # LONG regimes: moderate
        if direction == "LONG":
            base_size = min(base_size, 1.25)
        else:
            base_size = min(base_size, 1.0)
    else:
        base_size = min(base_size, 1.0)
    
    return base_size

# ========== UTILITY FUNCTIONS ==========
def to_utc(ts):
    """Ensure timestamp is UTC"""
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def rolling_zscore_safe(series, window=90):
    """FIXED: Shift AFTER calculation to prevent leakage"""
    return ((series - series.rolling(window).mean()) / 
            series.rolling(window).std()).shift(1)

def rolling_feature_safe(series, window, func='mean'):
    """Safe rolling with shift"""
    if func == 'mean':
        return series.rolling(window).mean().shift(1)
    elif func == 'std':
        return series.rolling(window).std().shift(1)
    elif func == 'median':
        return series.rolling(window).median().shift(1)
    return series

# ========== PRICE NOT NEAR LOWS HELPER ==========
def price_not_near_lows(row, df, lookback=90, min_pct=0.25):
    """
    ✅ FIX 1: Require price to be above X percentile of recent range
    Only applies to LONG positions
    """
    if pd.isna(row['eth_price']):
        return False
    
    idx = row.name
    start_idx = max(0, idx - lookback)
    recent_prices = df.iloc[start_idx:idx]['eth_price'].values
    
    if len(recent_prices) < 10:
        return True  # Not enough data
    
    price_min = np.min(recent_prices)
    price_max = np.max(recent_prices)
    
    if price_max - price_min < 1e-9:
        return True
    
    pct = (row['eth_price'] - price_min) / (price_max - price_min)
    return pct >= min_pct

# ========== PRICE NOT NEAR HIGHS HELPER ==========
def price_not_near_highs(row, df, lookback=90, max_pct=0.75):
    """
    Protection for LONG entries against buying local tops
    Only applies to LONG positions
    """
    if pd.isna(row['eth_price']):
        return True
    
    idx = row.name
    start_idx = max(0, idx - lookback)
    recent_prices = df.iloc[start_idx:idx]['eth_price'].values
    
    if len(recent_prices) < 10:
        return True
    
    price_min = np.min(recent_prices)
    price_max = np.max(recent_prices)
    
    if price_max - price_min < 1e-9:
        return True
    
    pct = (row['eth_price'] - price_min) / (price_max - price_min)
    return pct <= max_pct

# ========== DATA LOADING FROM FILES ==========
def load_data_from_files():
    """
    Load data from files saved by data_loader.py
    """
    print("📂 Loading data from saved files...")
    
    files_to_load = {
        'whale_data': 'data/whale_ml_ready.csv',
        'market_intent': 'data/market_intent_ml_ready.csv',
        'btc_price': 'data/price_cache/btc.csv',
        'eth_price': 'data/price_cache/eth.csv'
    }
    
    loaded_data = {}
    
    for name, filepath in files_to_load.items():
        if os.path.exists(filepath):
            try:
                if 'price' in name:
                    df = pd.read_csv(filepath, parse_dates=["date"])
                    df["date"] = df["date"].apply(to_utc)
                else:
                    df = pd.read_csv(filepath, parse_dates=["block_date"])
                    df["block_date"] = df["block_date"].apply(to_utc)
                
                loaded_data[name] = df
                print(f"✅ Loaded {name}: {len(df)} rows")
                print(f"   Date range: {df.iloc[0]['date' if 'price' in name else 'block_date'].date()} to "
                      f"{df.iloc[-1]['date' if 'price' in name else 'block_date'].date()}")
            except Exception as e:
                print(f"❌ Failed to load {name}: {e}")
                loaded_data[name] = pd.DataFrame()
        else:
            print(f"❌ {name} file not found: {filepath}")
            print(f"   Please run data_loader.py first to fetch data")
            loaded_data[name] = pd.DataFrame()
    
    # Check if we have all data
    if all(len(df) > 0 for df in loaded_data.values()):
        print(f"\n✅ All data loaded successfully")
    else:
        print(f"\n⚠️  Some data files are missing or empty")
        print(f"   Please run data_loader.py to fetch fresh data")
    
    return (
        loaded_data.get('whale_data', pd.DataFrame()),
        loaded_data.get('market_intent', pd.DataFrame()),
        loaded_data.get('btc_price', pd.DataFrame()),
        loaded_data.get('eth_price', pd.DataFrame())
    )

# ========== FEATURE ENGINEERING ==========
def add_price_features(df, price_col, prefix):
    """Add technical features for a price series"""
    df = df.copy()
    
    # Log returns
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    # Lagged returns (including lag 2 for LONG confirmation)
    for lag in [1, 2, 3, 7]:
        df[f'{prefix}_ret_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    # Volatility
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7).std().shift(1)
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30).std().shift(1)
    
    # RSI
    returns = df[f'{prefix}_log_return']
    gains = returns.where(returns > 0, 0).rolling(14).mean()
    losses = -returns.where(returns < 0, 0).rolling(14).mean()
    df[f'{prefix}_rsi'] = (100 - (100 / (1 + gains / (losses + 1e-10)))).shift(1)
    
    return df

def engineer_features(df_whales, df_market_intent, df_btc, df_eth):
    """Engineer all features with FIX A applied"""
    print("🔧 Engineering features...")
    
    # Merge price data
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    
    # Merge with whale data
    df = pd.merge(
        df_whales, 
        df_prices, 
        left_on='block_date', 
        right_on='date', 
        how='left'
    ).drop(columns=['date'])
    
    # Merge with market intent data
    df = pd.merge(
        df, 
        df_market_intent, 
        on='block_date', 
        how='left', 
        suffixes=('', '_intent')
    )
    
    df = df.sort_values('block_date').reset_index(drop=True)
    
    # Add price features (includes eth_ret_lag2)
    df = add_price_features(df, 'eth_price', 'eth')
    df = add_price_features(df, 'btc_price', 'btc')
    
    # ETH/BTC ratio features
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7).mean().shift(1)
    df['eth_btc_corr_30d'] = df['eth_log_return'].shift(1).rolling(30) \
        .corr(df['btc_log_return'].shift(1)).shift(1)
    
    # Apply safe rolling z-scores
    zscore_pairs = [
        ('whale_tx_count', 'whale_tx_zscore_90d'),
        ('tx_per_active', 'tx_per_active_zscore_90d'),
        ('eth_burned', 'eth_burned_zscore_90d'),
        ('exchange_volume', 'exchange_volume_zscore'),
    ]
    
    for raw_col, zscore_col in zscore_pairs:
        if raw_col in df.columns:
            df[zscore_col] = rolling_zscore_safe(df[raw_col], 90)
    
    # Burn/issuance ratio
    if all(col in df.columns for col in ['eth_burned', 'total_gas_fees']):
        df['burn_issuance_ratio'] = (df['eth_burned'] / (df['total_gas_fees'] + 1e-10)).shift(1)
    
    # Whale volume deltas
    if 'whale_volume_ratio' in df.columns:
        df['whale_volume_ratio_delta_1d'] = df['whale_volume_ratio'].diff(1).shift(1)
        df['whale_volume_ratio_delta_3d'] = df['whale_volume_ratio'].diff(3).shift(1)
    
    # Ensure ALL LONG_FEATURES exist
    for feature in LONG_FEATURES:
        if feature not in df.columns:
            print(f"⚠️  Creating missing LONG feature: {feature}")
            df[feature] = 0.0
    
    # Ensure ALL SHORT_FEATURES exist  
    for feature in SHORT_FEATURES:
        if feature not in df.columns:
            print(f"⚠️  Creating missing SHORT feature: {feature}")
            df[feature] = 0.0
    
    # Clean up intermediate columns
    df = df.drop(columns=['eth_log_return', 'btc_log_return'], errors='ignore')
    
    # Save engineered features
    df.to_csv('data/features_engineered.csv', index=False)
    print(f"✅ Features engineered: {len(df.columns)} columns, {len(df)} rows")
    print(f"   LONG features available: {sum(1 for f in LONG_FEATURES if f in df.columns)}/{len(LONG_FEATURES)}")
    print(f"   SHORT features available: {sum(1 for f in SHORT_FEATURES if f in df.columns)}/{len(SHORT_FEATURES)}")
    
    return df

# ========== TARGET CREATION ==========
def create_targets_two_tier(df, k=1.5):
    """
    Create two-tier SHORT labels (crash + breakdown)
    """
    print("🎯 Creating two-tier targets...")
    
    df = df.sort_values('block_date').reset_index(drop=True).copy()
    
    # Calculate returns
    df['eth_log_return'] = np.log(df['eth_price'] / df['eth_price'].shift(1))
    df['rolling_vol_30'] = df['eth_log_return'].rolling(30, min_periods=10).std()
    
    # T+2 returns and threshold
    df['return_t2'] = df['eth_log_return'].rolling(2).sum().shift(-2)
    
    # Dynamic threshold using 65th percentile
    df['threshold_t2'] = df['rolling_vol_30'].rolling(60, min_periods=20).quantile(0.65)
    
    # Tier 1: Crash (hard down)
    hard_down = (
        (df['return_t2'] < -df['threshold_t2']) &
        (df['eth_vol7'] > df['eth_vol30']).fillna(False)
    )
    
    # Tier 2: Breakdown (pre-crash)
    exchange_flow_median = df['exchange_flow_share'].rolling(90, min_periods=30).median()
    
    soft_down = (
        (df['eth_ret_lag1'].fillna(0) < 0) &
        (df['btc_ret_lag1'].fillna(0) < 0) &
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Create targets
    df['target_t2'] = 0
    df.loc[df['return_t2'] > df['threshold_t2'], 'target_t2'] = 1  # UP
    df.loc[hard_down | soft_down, 'target_t2'] = -1  # DOWN (both tiers)
    
    # Create binary targets
    df['y_long_t2'] = (df['target_t2'] == 1).astype(int)
    df['y_short_t2'] = (df['target_t2'] == -1).astype(int)
    
    # Clean up
    df = df.drop(columns=['eth_log_return'], errors='ignore')
    
    # Print distribution
    print("\n📊 Target Distribution (Two-Tier SHORT):")
    for state, label in [(-1, 'DOWN'), (0, 'FLAT'), (1, 'UP')]:
        count = (df['target_t2'] == state).sum()
        percentage = count / len(df) * 100
        print(f"  {label:5s}: {count:4d} ({percentage:5.1f}%)")
    
    hard_count = hard_down.sum()
    soft_count = soft_down.sum()
    total_down = (df['target_t2'] == -1).sum()
    
    print(f"\n  Tier 1 (crash):     {hard_count:4d}")
    print(f"  Tier 2 (breakdown): {soft_count:4d}")
    print(f"  Total DOWN:         {total_down:4d}")
    
    return df

# ========== REGIME DEFINITION ==========
def define_regimes_extended(df):
    """Define trading regimes including R5 distribution regime"""
    print("📈 Defining extended regimes...")
    
    if 'btc_ret_lag1' not in df.columns or 'eth_vol7' not in df.columns:
        df['regime_code'] = 'R0'
        return df
    
    # Standard regimes based on BTC trend and ETH volatility
    btc_trend_7d = df['btc_ret_lag1'].rolling(7).mean()
    df['btc_regime'] = pd.cut(
        btc_trend_7d, 
        bins=[-np.inf, -0.005, 0.005, np.inf], 
        labels=['DOWN', 'FLAT', 'UP']
    )
    
    vol_median = df['eth_vol7'].rolling(180, min_periods=60).median()
    df['vol_regime'] = (df['eth_vol7'] > vol_median).map({True: 'HIGH', False: 'LOW'})
    
    # Combine for standard regimes
    df['regime'] = df['btc_regime'].astype(str) + '_' + df['vol_regime'].astype(str)
    regime_map = {
        'UP_HIGH': 'R1',    # Bull high vol
        'UP_LOW': 'R2',     # Bull low vol
        'DOWN_HIGH': 'R3',  # Bear high vol
        'DOWN_LOW': 'R4',   # Bear low vol
    }
    df['regime_code'] = df['regime'].map(regime_map).fillna('R0')
    
    # R5: Whale distribution regime
    exchange_flow_median = df['exchange_flow_share'].rolling(60, min_periods=20).median()
    
    df['dist_regime'] = (
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Override with R5 where distribution regime is active
    df.loc[df['dist_regime'], 'regime_code'] = 'R5'
    
    # Print regime distribution
    print("\n📊 Extended Regime Distribution:")
    regime_stats = []
    for code in ['R1', 'R2', 'R3', 'R4', 'R5', 'R0']:
        count = (df['regime_code'] == code).sum()
        if len(df) > 0:
            pct = count / len(df) * 100
            icon = '🟢' if code == 'R1' else ('🔴' if code in ['R3', 'R5'] else '⚪')
            regime_stats.append(f"{icon} {code}: {count:4d} ({pct:5.1f}%)")
    
    # Print in two columns
    for i in range(0, len(regime_stats), 2):
        row = regime_stats[i:i+2]
        print("  " + " | ".join(row))
    
    return df

# ========== BUILD COMPLETE PIPELINE ==========
def build_pipeline_complete(df_features):
    """
    Create the complete pipeline dataset with features, targets, and regimes
    """
    print("\n" + "="*70)
    print("BUILDING COMPLETE PIPELINE DATASET")
    print("="*70)
    
    # Create targets
    df_with_targets = create_targets_two_tier(df_features)
    
    # Define regimes
    df_complete = define_regimes_extended(df_with_targets)
    
    # Ensure all required features exist
    for feature in LONG_FEATURES + SHORT_FEATURES:
        if feature not in df_complete.columns:
            df_complete[feature] = 0.0
    
    # Fill NaN values for features
    feature_cols = [col for col in df_complete.columns if col not in 
                   ['block_date', 'target_t2', 'y_long_t2', 'y_short_t2', 
                    'regime_code', 'btc_regime', 'vol_regime', 'regime', 'dist_regime']]
    
    df_complete[feature_cols] = df_complete[feature_cols].fillna(method='ffill').fillna(0)
    
    # Save complete pipeline
    df_complete.to_csv('data/pipeline_complete.csv', index=False)
    
    # Report statistics
    print(f"\n✅ Pipeline complete saved:")
    print(f"   Rows: {len(df_complete)}")
    print(f"   Columns: {len(df_complete.columns)}")
    print(f"   Date range: {df_complete['block_date'].min().date()} to {df_complete['block_date'].max().date()}")
    print(f"   File: data/pipeline_complete.csv")
    
    # Feature availability report
    print(f"\n📊 Feature Availability:")
    print(f"   LONG features: {sum(1 for f in LONG_FEATURES if f in df_complete.columns)}/{len(LONG_FEATURES)}")
    print(f"   SHORT features: {sum(1 for f in SHORT_FEATURES if f in df_complete.columns)}/{len(SHORT_FEATURES)}")
    
    # Check for missing features
    missing_long = [f for f in LONG_FEATURES if f not in df_complete.columns]
    missing_short = [f for f in SHORT_FEATURES if f not in df_complete.columns]
    
    if missing_long:
        print(f"   ⚠️  Missing LONG features: {missing_long}")
    if missing_short:
        print(f"   ⚠️  Missing SHORT features: {missing_short}")
    
    return df_complete

# ========== SHORT-SPECIFIC LOGIC ==========
def check_r3_short_allowed(row):
    """
    R3 short philosophy (early weakness only)
    """
    # Small red, not dump
    small_red = (-0.015 < row.get('eth_ret_lag1', 0) < 0)
    
    # BTC weakening
    btc_weak = (row.get('btc_ret_lag3', 0) < 0)
    
    # NOT vol expansion (early, not panic)
    eth_vol7 = row.get('eth_vol7', 0)
    eth_vol30 = row.get('eth_vol30', 1)
    no_vol_expansion = (eth_vol7 <= eth_vol30)
    
    # Whales increasing activity
    whale_activity = (row.get('whale_volume_ratio_delta_3d', 0) > 0)
    
    return small_red and btc_weak and no_vol_expansion and whale_activity

def calculate_short_veto_score(row):
    """Calculate veto scores for SHORT positions"""
    veto = 0
    reasons = []
    
    structural_score = 0
    flow_score = 0
    context_score = 0
    
    # Flow vetoes
    if row.get('net_exchange_flow_ratio', 0) < 0 and row.get('exchange_volume_zscore', 0) > 0:
        veto += 1
        flow_score += 1
        reasons.append('net_flow_negative_with_liquidity')
    
    if row.get('whale_exchange_flow_ratio', 0) > 0.6:
        veto += 1
        flow_score += 1
        reasons.append('whale_to_exchange')
    
    # Structural vetoes
    if row.get('btc_ret_lag1', 0) < -0.02 and row.get('eth_ret_lag1', 0) < -0.01:
        veto += 2
        structural_score += 2
        reasons.append('btc_breakdown')
    
    if row.get('eth_vol7', 0) > row.get('eth_vol30', 0):
        veto += 2
        structural_score += 2
        reasons.append('vol_expansion')
    
    # Context vetoes
    if row.get('btc_ret_lag1', 0) > 0.02:
        veto += 1
        context_score += 1
        reasons.append('btc_conflict')
    
    if row.get('eth_vol7', 0) < row.get('eth_vol30', 0) * 0.7:
        veto += 1
        context_score += 1
        reasons.append('low_volatility')
    
    return veto, structural_score, flow_score, context_score, reasons

def check_short_requirements(row, regime, structural_score, flow_score):
    """Check SHORT-specific requirements with nuanced flow confirmation"""
    reasons = []
    
    # Flow confirmation with nuance
    if flow_score == 0:
        # Allow only if structural weakness + bearish BTC context
        if not (
            structural_score > 0 and
            row.get('btc_ret_lag1', 0) < 0
        ):
            reasons.append("no_flow_confirmation")
    
    # R5 stronger flow requirement (only if we have flow signals)
    if regime == "R5" and flow_score > 0 and flow_score < 2:
        reasons.append("weak_distribution_flow")
    
    # Structural check - no structural weakness = no short
    if structural_score == 0:
        reasons.append("no_structural_break")
    
    # R3 short check (using new philosophy)
    if regime == "R3" and not check_r3_short_allowed(row):
        reasons.append("r3_no_early_weakness")
    
    return reasons

# ========== LONG-SPECIFIC LOGIC ==========
def long_veto(row):
    """LONG veto - minimal and asymmetric"""
    veto = []
    
    if row.get('btc_ret_lag1', 0) < -0.02:
        veto.append("btc_drawdown")
    
    if row.get('whale_exchange_flow_ratio', 0) > 0.6:
        veto.append("distribution")
    
    if row.get('eth_vol7', 0) > row.get('eth_vol30', 0) * 1.5:
        veto.append("vol_spike")
    
    return veto

def long_confirmation(row):
    """
    LONG confirmation logic
    Stage A: ML finds accumulation
    Stage B: Confirm price is responding
    """
    return (
        row.get('eth_ret_lag1', 0) > 0 and
        row.get('eth_ret_lag2', 0) > 0 and
        row.get('eth_vol7', 0) < row.get('eth_vol30', 1)
    )

# ========== MODEL MANAGEMENT ==========

def rebuild_models_if_needed(df_pipeline):
    """
    Rebuild models if they don't exist or feature mismatch
    Returns: (short_model, long_model)
    """
    short_model = None
    long_model = None
    
    # Check SHORT model
    short_model_path = 'models/r5_short_final.pkl'
    if os.path.exists(short_model_path):
        try:
            short_model = joblib.load(short_model_path)
            print(f"✅ SHORT model loaded from {short_model_path}")
            
            # Check if model has feature_names_ attribute
            if hasattr(short_model, 'feature_names_'):
                print(f"   Model expects {len(short_model.feature_names_)} features")
                missing = [f for f in short_model.feature_names_ if f not in df_pipeline.columns]
                if missing:
                    print(f"⚠️  SHORT model missing features: {missing[:5]}")
                    print("   Rebuilding SHORT model...")
                    short_model = None
            else:
                print("⚠️  SHORT model missing feature_names_, rebuilding...")
                short_model = None
        except Exception as e:
            print(f"⚠️  Error loading SHORT model: {e}")
            short_model = None
    
    # Check LONG model
    long_model_path = 'models/R1_R2_LONG.pkl'
    if os.path.exists(long_model_path):
        try:
            long_model = joblib.load(long_model_path)
            print(f"✅ LONG model loaded from {long_model_path}")
            
            # Check if model has feature_names_ attribute
            if hasattr(long_model, 'feature_names_'):
                print(f"   Model expects {len(long_model.feature_names_)} features")
                missing = [f for f in long_model.feature_names_ if f not in df_pipeline.columns]
                if missing:
                    print(f"⚠️  LONG model missing features: {missing[:5]}")
                    print("   Rebuilding LONG model...")
                    long_model = None
            else:
                print("⚠️  LONG model missing feature_names_, rebuilding...")
                long_model = None
        except Exception as e:
            print(f"⚠️  Error loading LONG model: {e}")
            long_model = None
    
    # Rebuild SHORT model if needed
    if short_model is None:
        print("\n" + "="*70)
        print("REBUILDING SHORT MODEL (R5)")
        print("="*70)
        
        df_r5 = df_pipeline[df_pipeline['regime_code'] == 'R5'].copy()
        
        # ✅ CRITICAL FIX 1: Use only features that exist in the data
        short_features = [f for f in SHORT_FEATURES if f in df_r5.columns]
        print(f"   Using {len(short_features)} SHORT features: {short_features}")
        
        if len(df_r5) >= 50:
            split_idx = int(len(df_r5) * 0.8)
            X_train_short = df_r5[short_features].iloc[:split_idx].fillna(0)
            y_train_short = df_r5['y_short_t2'].iloc[:split_idx]
            
            print(f"   Training on {len(X_train_short)} R5 samples")
            
            short_model = GradientBoostingClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42
            )
            short_model.fit(X_train_short, y_train_short)
            
            # ✅ CRITICAL FIX 2: Store feature names in the model
            short_model.feature_names_ = short_features
            joblib.dump(short_model, short_model_path)
            
            # Verify
            print(f"✅ SHORT model rebuilt and saved")
            print(f"   Model now expects {len(short_model.feature_names_)} features")
            print(f"   Features: {short_model.feature_names_}")
        else:
            print("⚠️  Insufficient R5 data for SHORT model")
    
    # Rebuild LONG model if needed
    if long_model is None:
        print("\n" + "="*70)
        print("REBUILDING LONG MODEL (R1+R2)")
        print("="*70)
        
        df_long = df_pipeline[df_pipeline['regime_code'].isin(['R1', 'R2'])].copy()
        
        # Remove obvious traps
        df_long = df_long[
            (df_long['btc_ret_lag1'] > -0.02) &
            (df_long['whale_exchange_flow_ratio'] < 0.6)
        ]
        
        # ✅ CRITICAL FIX 3: Use only features that exist in the data
        long_features = [f for f in LONG_FEATURES if f in df_long.columns]
        print(f"   Using {len(long_features)} LONG features: {long_features}")
        
        X_long = df_long[long_features].fillna(0)
        y_long = df_long['y_long_t2']
        
        if len(X_long) >= 50:
            long_model = GradientBoostingClassifier(
                n_estimators=120,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                random_state=42
            )
            
            long_model.fit(X_long, y_long)
            
            # ✅ CRITICAL FIX 4: Store feature names in the model
            long_model.feature_names_ = long_features
            joblib.dump(long_model, long_model_path)
            
            print("✅ LONG model rebuilt and saved")
            print(f"   Model now expects {len(long_model.feature_names_)} features")
            print(f"   Features: {long_model.feature_names_}")
            
            # Basic validation
            probs = long_model.predict_proba(X_long)[:, 1]
            preds = (probs >= 0.60).astype(int)
            prec = precision_score(y_long, preds, zero_division=0)
            rec = recall_score(y_long, preds, zero_division=0)
            
            print(f"   Training precision: {prec:.3f}")
            print(f"   Training recall: {rec:.3f}")
        else:
            print("⚠️  Insufficient LONG data")
    
    return short_model, long_model

# ========== UNIFIED SIGNAL GENERATION ==========

def generate_unified_signal(row, df, short_model, long_model):
    """
    UNIFIED SIGNAL GENERATION with asymmetric LONG confirmation
    """
    regime = row.get('regime_code', 'R0')
    date_str = str(row['block_date'].date()) if 'block_date' in row else str(row.name)
    
    # Base signal object (will be completed)
    signal = {
        "date": date_str,
        "regime": regime,
        "direction": None,
        "model_probability": 0.0,
        "adjusted_confidence": 0.0,
        "position_size": 0.0,
        "reasons": [],
        "action": "NO_TRADE"
    }
    
    # ===== LONG LOGIC (R1/R2/R3) =====
    if regime in ['R1', 'R2', 'R3'] and long_model:  # Note: Now includes R3 for completeness
        signal["direction"] = "LONG"
        
        # Use model's stored feature names for inference
        if not hasattr(long_model, 'feature_names_'):
            signal["reasons"] = ["model_error: long_model missing feature_names_"]
            return signal
        
        features = long_model.feature_names_
        try:
            # Ensure features exist in row and fill missing with 0
            X = row.reindex(features, fill_value=0).values.reshape(1, -1)
            prob = long_model.predict_proba(X)[0, 1]
            signal["model_probability"] = float(prob)
        except Exception as e:
            signal["reasons"] = [f"model_error: {str(e)[:100]}"]
            return signal
        
        # Step 1: Check probability threshold (LOWER for LONG)
        if prob < LONG_ENTRY_THRESHOLD:
            signal["reasons"] = ["low_model_probability"]
            return signal
        
        # ✅ FIXED: Step 2: Apply explicit regime-aware asymmetric confirmation
        confirm_score = 0
        
        if row.get('eth_ret_lag1', 0) > 0:
            confirm_score += 1
        if row.get('eth_ret_lag2', 0) > 0:
            confirm_score += 1
        if row.get('eth_vol7', 0) < row.get('eth_vol30', 1):
            confirm_score += 1
        
        # ✅ EXPLICIT regime thresholds
        if regime == "R1":
            required_score = 1  # Early bull: allow early signs
        elif regime == "R2":
            required_score = 2  # Late bull: require agreement
        elif regime == "R3":
            required_score = 3  # Early bear: extremely strict
        else:
            required_score = 2  # Safe default
        
        if confirm_score < required_score:
            signal["reasons"] = [f"weak_price_confirmation ({confirm_score}/{required_score})"]
            return signal
        
        # Step 3: Price not near highs (LONG protection) - still applies to all regimes
        if not price_not_near_highs(row, df, lookback=90, max_pct=0.75):
            signal["reasons"] = ["price_near_highs"]
            return signal
        
        # Step 4: Apply LONG vetoes
        veto_reasons = long_veto(row)
        if veto_reasons:
            signal["reasons"] = veto_reasons
            return signal
        
        # Step 5: All checks passed - calculate confidence and size
        veto_score = len(veto_reasons)
        signal["adjusted_confidence"] = adjust_confidence_unified(
            prob, regime, direction="LONG", veto_score=veto_score
        )
        
        # Step 6: Regime-aware confidence floor
        if regime == "R1":
            confidence_floor = 0.50  # Lower for early bull
        elif regime == "R2":
            confidence_floor = 0.55  # Higher for late bull
        elif regime == "R3":
            confidence_floor = 0.60  # Highest for early bear (should be rare)
        else:
            confidence_floor = 0.55  # Default
        
        if signal["adjusted_confidence"] < confidence_floor:
            signal["reasons"] = ["low_final_confidence"]
            signal["direction"] = None
            return signal
        
        # ✅ FIXED: Use the updated position sizing function
        signal["position_size"] = map_confidence_to_size_unified(
            signal["adjusted_confidence"], regime, direction="LONG"
        )
        signal["reasons"] = ["ml_accumulation", "price_confirmation"]
        signal["action"] = "ENTER"
        return signal
    
    # ===== SHORT LOGIC (R3/R5) =====
    elif regime in ['R3', 'R5'] and short_model:
        signal["direction"] = "SHORT"
        
        # Use model's stored feature names for inference
        if not hasattr(short_model, 'feature_names_'):
            signal["reasons"] = ["model_error: short_model missing feature_names_"]
            return signal
        
        features = short_model.feature_names_
        try:
            # Ensure features exist in row and fill missing with 0
            X = row.reindex(features, fill_value=0).values.reshape(1, -1)
            prob = short_model.predict_proba(X)[0, 1]
            signal["model_probability"] = float(prob)
        except Exception as e:
            signal["reasons"] = [f"model_error: {str(e)[:100]}"]
            return signal
        
        # Step 1: Check probability threshold (HIGHER for SHORT)
        if prob < SHORT_ENTRY_THRESHOLD:
            signal["reasons"] = ["low_model_probability"]
            return signal
        
        # Step 2: Calculate veto scores
        veto, structural_score, flow_score, context_score, veto_reasons = calculate_short_veto_score(row)
        
        # Step 3: Check SHORT-specific requirements (with fixed flow logic)
        requirement_failures = check_short_requirements(row, regime, structural_score, flow_score)
        if requirement_failures:
            signal["reasons"] = requirement_failures
            return signal
        
        # Step 4: All checks passed - calculate confidence and size
        signal["adjusted_confidence"] = adjust_confidence_unified(
            prob, regime, direction="SHORT", veto_score=veto
        )
        
        # Final confidence check
        if signal["adjusted_confidence"] < 0.55:
            signal["reasons"] = ["low_final_confidence"]
            signal["direction"] = None
            return signal
        
        signal["position_size"] = map_confidence_to_size_unified(
            signal["adjusted_confidence"], regime, direction="SHORT"
        )
        signal["reasons"] = veto_reasons  # Use veto reasons as trade reasons
        signal["action"] = "ENTER"
        return signal
    
    # ===== NEUTRAL REGIME =====
    else:
        signal["reasons"] = ["neutral_regime"]
        return signal
            
def generate_daily_signal_unified(df, short_model, long_model):
    """
    Generate unified daily signal (uses latest row)
    """
    latest_row = df.iloc[-1].copy()
    return generate_unified_signal(latest_row, df, short_model, long_model)

# ========== SIGNAL INSPECTION ==========
def inspect_signals_unified(df, short_model, long_model, num_signals=60):
    """
    Inspect signals with unified logic
    """
    print("\n" + "="*70)
    print(f"UNIFIED SIGNAL INSPECTION (Last {num_signals} days)")
    print("="*70)
    
    recent_data = df.iloc[-num_signals:].copy()
    print(f"Date range: {recent_data['block_date'].min().date()} to {recent_data['block_date'].max().date()}")
    
    signals = []
    long_count = 0
    short_count = 0
    no_trade_count = 0
    
    for idx, row in recent_data.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
        
        # Print only trade signals
        if signal['action'] == 'ENTER':
            if signal['direction'] == 'LONG':
                long_count += 1
                print(f"\n🟢 LONG:  {signal['date']} | Regime: {signal['regime']} | "
                      f"Conf: {signal['adjusted_confidence']:.2f} | Size: {signal['position_size']:.2f}")
            elif signal['direction'] == 'SHORT':
                short_count += 1
                print(f"\n🔴 SHORT: {signal['date']} | Regime: {signal['regime']} | "
                      f"Conf: {signal['adjusted_confidence']:.2f} | Size: {signal['position_size']:.2f}")
        else:
            no_trade_count += 1
    
    print(f"\n📊 Signal Summary:")
    print(f"   Total days: {len(signals)}")
    print(f"   LONG signals: {long_count} ({long_count/len(signals)*100:.1f}%)")
    print(f"   SHORT signals: {short_count} ({short_count/len(signals)*100:.1f}%)")
    print(f"   NO_TRADE: {no_trade_count} ({no_trade_count/len(signals)*100:.1f}%)")
    
    # Regime distribution
    print(f"\n📈 Regime Distribution:")
    regime_dist = {}
    for signal in signals:
        regime = signal.get('regime', 'UNKNOWN')
        regime_dist[regime] = regime_dist.get(regime, 0) + 1
    
    for regime in sorted(regime_dist.keys()):
        count = regime_dist[regime]
        print(f"   {regime}: {count} days ({count/len(signals)*100:.1f}%)")
    
    # Rejection reasons analysis
    print(f"\n🔍 Rejection Reasons:")
    rejection_reasons = {}
    for signal in signals:
        if signal['action'] == 'NO_TRADE' and signal.get('reasons'):
            for reason in signal['reasons']:
                rejection_reasons[reason] = rejection_reasons.get(reason, 0) + 1
    
    for reason, count in sorted(rejection_reasons.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"   {reason}: {count} times")
    
    return signals
def inspect_long_signals_bull_cycle(df, short_model, long_model, start_date='2020-01-01', end_date='2022-01-01'):
    """
    Inspect LONG signals during the 2020-2021 bull cycle
    """
    print("\n" + "="*70)
    print(f"LONG SIGNAL INSPECTION: {start_date} to {end_date}")
    print("="*70)
    
    # Filter to bull cycle period
    mask = (df['block_date'] >= start_date) & (df['block_date'] <= end_date)
    bull_data = df[mask].copy()
    
    print(f"Period: {bull_data['block_date'].min().date()} to {bull_data['block_date'].max().date()}")
    print(f"Total days: {len(bull_data)}")
    
    # Get R1/R2 days
    bull_regimes = bull_data[bull_data['regime_code'].isin(['R1', 'R2'])].copy()
    print(f"R1/R2 days: {len(bull_regimes)}")
    print(f"  R1: {(bull_regimes['regime_code'] == 'R1').sum()} days")
    print(f"  R2: {(bull_regimes['regime_code'] == 'R2').sum()} days")
    
    # Generate signals for R1/R2 days only
    long_signals = []
    for idx, row in bull_regimes.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        if signal['direction'] == 'LONG':
            long_signals.append(signal)
    
    # Analyze LONG signals
    print(f"\n📊 LONG Signal Analysis:")
    print(f"  Total LONG signals: {len(long_signals)}")
    
    if len(long_signals) > 0:
        df_long_signals = pd.DataFrame(long_signals)
        
        # Group by regime
        regime_counts = df_long_signals['regime'].value_counts()
        for regime, count in regime_counts.items():
            print(f"  {regime}: {count} signals")
        
        # Analyze timing (early vs late bull)
        df_long_signals['date_dt'] = pd.to_datetime(df_long_signals['date'])
        df_long_signals['month'] = df_long_signals['date_dt'].dt.to_period('M')
        monthly_counts = df_long_signals['month'].value_counts().sort_index()
        
        print(f"\n📅 Monthly distribution:")
        for month, count in monthly_counts.head(12).items():  # Show first 12 months
            print(f"  {month}: {count} signals")
        
        # Check if signals avoid tops
        print(f"\n🔍 Top avoidance check:")
        for signal in df_long_signals.head(5).to_dict('records'):  # Show first 5 signals
            date_str = signal['date']
            row = bull_data[bull_data['block_date'] == pd.Timestamp(date_str)]
            if not row.empty:
                row = row.iloc[0]
                # Check if price near highs (should be False for good LONGs)
                near_highs = not price_not_near_highs(row, df, lookback=90, max_pct=0.75)
                status = "⚠️ NEAR HIGHS" if near_highs else "✅ NOT NEAR HIGHS"
                print(f"  {date_str}: {status} | Price: ${row['eth_price']:.0f} | Conf: {signal['adjusted_confidence']:.2f}")
    
    # Also check how many R1/R2 days were rejected and why
    print(f"\n🔍 LONG Rejection Analysis (R1/R2 days):")
    rejection_counts = {}
    for idx, row in bull_regimes.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        if signal['action'] != 'ENTER' and signal['direction'] == 'LONG':
            for reason in signal['reasons']:
                rejection_counts[reason] = rejection_counts.get(reason, 0) + 1
    
    for reason, count in sorted(rejection_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {reason}: {count} times")
    
    return long_signals

# ========== MAIN PIPELINE ==========
def run_unified_pipeline():
    """
    Execute complete pipeline with UNIFIED signal contract
    """
    print("\n" + "="*70)
    print("ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT")
    print("="*70)
    print("✅ Features FROZEN")
    print("✅ Regimes FROZEN")
    print("✅ Unified confidence & sizing")
    print("="*70)
    
    # Step 1-3: Load data, engineer features, build pipeline
    df_whales, df_market, df_btc, df_eth = load_data_from_files()
    
    if any(d.empty for d in [df_whales, df_market, df_btc, df_eth]):
        print("\n❌ Missing data. Run data_loader.py first.")
        return None, None, None
    
    df_features = engineer_features(df_whales, df_market, df_btc, df_eth)
    df_pipeline = build_pipeline_complete(df_features)
    
    # Step 4: Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df_pipeline)
    
    # Step 5: Generate live signal
    if short_model and long_model:
        print("\n" + "="*70)
        print("GENERATING UNIFIED LIVE SIGNAL")
        print("="*70)
        
        signal = generate_daily_signal_unified(df_pipeline, short_model, long_model)
        print(json.dumps(signal, indent=2))
        
        with open('data/latest_signal_unified.json', 'w') as f:
            json.dump(signal, f, indent=2)
    
    # Step 6: Inspect 60-day history
    if short_model and long_model:
        print("\n" + "="*70)
        print("60-DAY UNIFIED SIGNAL INSPECTION")
        print("="*70)
        
        signals = inspect_signals_unified(df_pipeline, short_model, long_model, 60)
        
        df_signals = pd.DataFrame(signals)
        df_signals.to_csv('validation/signal_inspection_unified.csv', index=False)
        print(f"\n✅ Saved: validation/signal_inspection_unified.csv")
    
    return df_pipeline, short_model, long_model

# ========== PAPER TRADE TEST ==========
def run_unified_paper_test():
    """
    Run 60-day paper trade test with UNIFIED logic
    """
    print("\n" + "="*70)
    print("60-DAY PAPER TRADE TEST (UNIFIED LOGIC)")
    print("="*70)
    
    # Load pipeline data
    if not os.path.exists('data/pipeline_complete.csv'):
        print("❌ Pipeline data not found. Run unified pipeline first.")
        return
    
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    df = df.sort_values('block_date')
    
    # Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df)
    
    if not short_model or not long_model:
        print("❌ Could not load or rebuild models")
        return
    
    # Take last 60 days
    test_period = df.iloc[-60:].copy()
    
    # Generate signals
    signals = []
    for i, row in test_period.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
        
        # Print trade signals
        if signal['action'] == 'ENTER':
            direction_icon = "🟢" if signal['direction'] == 'LONG' else "🔴"
            print(f"{direction_icon} {signal['date']}: {signal['action']} {signal['direction']} "
                  f"@ ${row['eth_price']:.0f} (conf: {signal['adjusted_confidence']:.2f}, "
                  f"size: {signal['position_size']:.2f})")
    
    # Analyze results
    df_signals = pd.DataFrame(signals)
    
    print(f"\n📊 Test Results (60 days):")
    print(f"   Total days: {len(df_signals)}")
    print(f"   ENTER signals: {(df_signals['action'] == 'ENTER').sum()}")
    print(f"   LONG signals: {(df_signals['direction'] == 'LONG').sum()}")
    print(f"   SHORT signals: {(df_signals['direction'] == 'SHORT').sum()}")
    
    # Manual review questions
    if (df_signals['action'] == 'ENTER').sum() > 0:
        print(f"\n🔍 Manual Review Questions:")
        print(f"   1. Do LONGs occur only in R1/R2?")
        print(f"   2. Do SHORTs occur only in R3/R5?")
        print(f"   3. Are LONGs buying strength, not tops?")
        print(f"   4. Are SHORTs selling weakness/distribution?")
        print(f"   5. Are position sizes reasonable for regime?")
    
    # Regime distribution
    print(f"\n📈 Regime Distribution:")
    regime_dist = test_period['regime_code'].value_counts()
    for regime, count in regime_dist.items():
        print(f"   {regime}: {count} days ({count/len(test_period)*100:.1f}%)")
    
    return df_signals

# ========== MANUAL SIGNAL REVIEW ==========
def manual_signal_review(num_days=30):
    """
    Manual review of signals with guided questions
    """
    print("\n" + "="*70)
    print(f"MANUAL SIGNAL REVIEW ({num_days} days)")
    print("="*70)
    
    # Load pipeline data
    if not os.path.exists('data/pipeline_complete.csv'):
        print("❌ Pipeline data not found. Run unified pipeline first.")
        return
    
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    df = df.sort_values('block_date')
    
    # Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df)
    
    if not short_model or not long_model:
        print("❌ Could not load or rebuild models")
        return
    
    # Generate signals
    recent_data = df.iloc[-num_days:].copy()
    signals = []
    
    for i, row in recent_data.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
    
    # Review template
    print("\n📝 Review Template (for each ENTER signal):")
    print("-" * 40)
    
    for signal in signals:
        if signal['action'] == 'ENTER':
            print(f"\n📅 {signal['date']} - {signal['direction']} in {signal['regime']}")
            print(f"   Confidence: {signal['adjusted_confidence']:.2f}")
            print(f"   Position Size: {signal['position_size']:.2f}")
            print(f"   Reasons: {', '.join(signal['reasons'])}")
            
            # Find the row data
            row = df[df['block_date'] == pd.Timestamp(signal['date'])]
            if not row.empty:
                row = row.iloc[0]
                print(f"   ETH Price: ${row['eth_price']:.0f}")
                print(f"   BTC Ret Lag1: {row.get('btc_ret_lag1', 0):.3f}")
                print(f"   ETH Ret Lag1: {row.get('eth_ret_lag1', 0):.3f}")
            
            # Review questions
            if signal['direction'] == 'LONG':
                print("   Questions:")
                print("   1. Is price breaking structure upward?")
                print("   2. Is BTC aligned or neutral?")
                print("   3. Are we buying strength, not tops?")
            else:
                print("   Questions:")
                print("   1. Is this distribution or panic?")
                print("   2. Is liquidity present?")
                print("   3. Is this early weakness (R3) or real distribution (R5)?")
            
            print("-" * 40)
    
    return signals

# ========== MAIN EXECUTION ==========
if __name__ == "__main__":
    print("\n" + "="*70)
    print("ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT")
    print("="*70)
    print("PHASE: EXECUTION HARDENING (Features FROZEN)")
    print(f"{'='*70}")
    
    # Check if data files exist
    required_files = [
        'data/whale_ml_ready.csv',
        'data/market_intent_ml_ready.csv', 
        'data/price_cache/btc.csv',
        'data/price_cache/eth.csv'
    ]
    
    missing_files = [f for f in required_files if not os.path.exists(f)]
    
    if missing_files:
        print(f"\n⚠️  Missing data files:")
        for f in missing_files:
            print(f"   - {f}")
        print(f"\nPlease run data_loader.py first to fetch data")
        print(f"Or place the required CSV files in the data directory")
        exit(1)
    
        # Ask user what to do
        print("\n📋 Available Options:")
        print("   1. Run unified pipeline (train models + generate signal)")
        print("   2. 60-day paper trade test (unified logic)")
        print("   3. Manual signal review (30 days)")
        print("   4. Inspect signals (60 days)")
        print("   5. Load and check data only")
        print("   6. Extended LONG inspection (2020-2021 bull cycle)")  # NEW OPTION
    
    choice = input("\nSelect option (1-6): ").strip()
    
    if choice == '1':
        df_pipeline, short_model, long_model = run_unified_pipeline()
        
        if df_pipeline is not None:
            print("\n✅ Unified pipeline complete")
            print("\n📋 Next steps:")
            print("   1. Review validation/signal_inspection_unified.csv")
            print("   2. Run option 2 for paper trade test")
            print("   3. Run option 3 for manual review")
    
    elif choice == '2':
        signals = run_unified_paper_test()
        
        print("\n📋 Review questions answered:")
        print("   ✅ LONGs only in R1/R2?")
        print("   ✅ SHORTs only in R3/R5?")
        print("   ✅ Position sizing consistent?")
        print("   ✅ Confidence ranges reasonable?")
    
    elif choice == '3':
        num_days = input("How many days to review? (default: 30): ").strip()
        try:
            num_days = int(num_days) if num_days else 30
        except:
            num_days = 30
        
        signals = manual_signal_review(num_days)
    
    elif choice == '4':
        # Load pipeline data
        if not os.path.exists('data/pipeline_complete.csv'):
            print("❌ Pipeline data not found. Run option 1 first.")
        else:
            df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
            short_model, long_model = rebuild_models_if_needed(df)
            
            if short_model and long_model:
                signals = inspect_signals_unified(df, short_model, long_model, 60)
    
    elif choice == '5':
        print("\n📂 Loading and checking data...")
        df_whales, df_market, df_btc, df_eth = load_data_from_files()
        
        print(f"\n✅ Data loaded successfully:")
        print(f"   Whale data: {len(df_whales)} rows")
        print(f"   Market data: {len(df_market)} rows")
        print(f"   BTC price: {len(df_btc)} rows")
        print(f"   ETH price: {len(df_eth)} rows")
    
    elif choice == '6':
        print("\n" + "="*70)
        print("EXTENDED LONG INSPECTION (2020-2021 BULL CYCLE)")
        print("="*70)
        
        # Load pipeline data
        if not os.path.exists('data/pipeline_complete.csv'):
            print("❌ Pipeline data not found. Run option 1 first.")
        else:
            df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
            df = df.sort_values('block_date')
            
            # Load models
            short_model_path = 'models/r5_short_final.pkl'
            long_model_path = 'models/R1_R2_LONG.pkl'
            
            if os.path.exists(short_model_path) and os.path.exists(long_model_path):
                short_model = joblib.load(short_model_path)
                long_model = joblib.load(long_model_path)
                
                # Ask for date range
                print("\n📅 Select inspection period:")
                print("   1. 2020-2021 bull cycle (2020-01-01 to 2022-01-01)")
                print("   2. Full available history")
                print("   3. Custom date range")
                
                period_choice = input("Select (1-3): ").strip()
                
                if period_choice == '1':
                    start_date = '2020-01-01'
                    end_date = '2022-01-01'
                elif period_choice == '2':
                    start_date = df['block_date'].min().strftime('%Y-%m-%d')
                    end_date = df['block_date'].max().strftime('%Y-%m-%d')
                elif period_choice == '3':
                    start_date = input("Start date (YYYY-MM-DD): ").strip()
                    end_date = input("End date (YYYY-MM-DD): ").strip()
                else:
                    start_date = '2020-01-01'
                    end_date = '2022-01-01'
                
                # Run inspection
                long_signals = inspect_long_signals_bull_cycle(
                    df, short_model, long_model, 
                    start_date=start_date, 
                    end_date=end_date
                )
                
                # Save results
                if long_signals:
                    df_results = pd.DataFrame(long_signals)
                    df_results.to_csv('validation/long_signals_bull_cycle.csv', index=False)
                    print(f"\n✅ Saved: validation/long_signals_bull_cycle.csv")
            else:
                print("❌ Models not found. Run option 1 first.")
    else:
        print("❌ Invalid option")
        
    print("\n" + "="*70)
    print("UNIFIED SIGNAL CONTRACT - SUMMARY")
    print("="*70)
    print("✅ Features: FROZEN")
    print("✅ Regimes: FROZEN")
    print("✅ Confidence: Unified (saturation + caps)")
    print("✅ Sizing: Unified (regime-aware caps)")
    print("✅ Signal Object: Consistent")
    print(f"{'='*70}")


ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT
PHASE: EXECUTION HARDENING (Features FROZEN)

ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT
✅ Features FROZEN
✅ Regimes FROZEN
✅ Unified confidence & sizing
📂 Loading data from saved files...
✅ Loaded whale_data: 3005 rows
   Date range: 2017-10-16 to 2026-01-06
✅ Loaded market_intent: 3005 rows
   Date range: 2017-10-16 to 2026-01-06
✅ Loaded btc_price: 4635 rows
   Date range: 2013-04-28 to 2026-01-06
✅ Loaded eth_price: 3805 rows
   Date range: 2015-08-07 to 2026-01-06

✅ All data loaded successfully
🔧 Engineering features...
⚠️  Creating missing SHORT feature: exchange_volume_zscore
✅ Features engineered: 54 columns, 3005 rows
   LONG features available: 16/16
   SHORT features available: 11/11

BUILDING COMPLETE PIPELINE DATASET
🎯 Creating two-tier targets...

📊 Target Distribution (Two-Tier SHORT):
  DOWN :  497 ( 16.5%)
  FLAT : 1915 ( 63.7%)
  UP   :  593 ( 19.7%)

  Tier 1 (crash):      228
  Tier 2 (breakdown):  302
  T

In [2]:
"""
ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT
FINAL HARDENING PHASE - ALL CRITICAL FIXES APPLIED
"""

import os
import time
import json
import warnings
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from dotenv import load_dotenv
import joblib

warnings.filterwarnings('ignore')
load_dotenv()

# Create directories
for d in ['validation', 'backtest', 'models']:
    os.makedirs(d, exist_ok=True)

# ========== CONFIGURATION ==========
# Feature sets 
SHORT_FEATURES = [
    'exchange_flow_share', 'net_exchange_flow_ratio', 'whale_exchange_flow_ratio',
    'whale_exchange_asymmetry', 'vol_ratio', 'btc_ret_lag1',  
    'btc_ret_lag3', 'eth_btc_corr_30d', 'whale_volume_ratio_delta_3d',
    'exchange_volume_zscore'
]

LONG_FEATURES = [
    'btc_rsi', 'vol_ratio', 'whale_volume_ratio', 'eth_rsi',  
    'btc_ret_lag1', 'eth_burned_zscore_90d', 'eth_btc_corr_30d',
    'eth_ret_lag1', 'btc_ret_lag7', 'btc_vol30',
    'whale_volume_ratio_delta_1d', 'whale_volume_ratio_delta_3d',
    'exchange_flow_share', 'net_exchange_flow_ratio',
    'whale_exchange_flow_ratio', 'tx_per_active_zscore_90d'
]

# Trading parameters
SLIPPAGE = 0.0008
FEES = 0.0004

# Entry thresholds - FROZEN (can tune later)
LONG_ENTRY_THRESHOLD = 0.50  # Lower for LONG to allow confirmation rescue
SHORT_ENTRY_THRESHOLD = 0.55  # Higher for SHORT (requires stronger signal)

# ========== UNIFIED CONFIDENCE & POSITION SIZING ==========

def adjust_confidence_unified(prob, regime, direction=None, veto_score=0, row=None):
    """
    UNIFIED CONFIDENCE ADJUSTMENT with optional funding modifier
    """
    # Base confidence from model
    base_conf = float(prob)
    
    # Apply veto boost (same for both directions)
    veto_boost = np.tanh(veto_score / 3) * 0.15
    
    # Initial adjusted confidence
    adj_conf = np.clip(base_conf + veto_boost, 0, 1)
    
    # Apply confidence caps based on regime
    if regime == "R3":
        max_conf = 0.70
    elif regime == "R5":
        max_conf = 0.85
    elif regime in ["R1", "R2"]:
        max_conf = 0.75
    else:
        max_conf = 0.95
    
    adj_conf = min(adj_conf, max_conf)
    
    # ====== OPTIONAL: Apply funding modifier (only if row provided) ======
    funding_reasons = []
    if row is not None and direction is not None:
        # This is optional - only applies in generate_unified_signal
        adj_conf, funding_reasons = apply_funding_modifier(row, direction, adj_conf)
    
    return adj_conf, funding_reasons

def map_confidence_to_size_unified(conf, regime=None, direction=None):
    """
    UNIFIED POSITION SIZING with explicit regime-aware thresholds
    """
    # ✅ EXPLICIT asymmetric confidence floors
    if direction == "LONG":
        if conf < 0.50:  # Lower threshold for LONG
            return 0.0
    elif direction == "SHORT":
        if conf < 0.55:  # Higher threshold for SHORT
            return 0.0
    else:
        if conf < 0.55:  # Default
            return 0.0
    
    # Base sizing scale (same for both directions)
    if conf < 0.60: 
        base_size = 0.25
    elif conf < 0.65: 
        base_size = 0.50
    elif conf < 0.70: 
        base_size = 0.75
    elif conf < 0.75: 
        base_size = 1.00
    elif conf < 0.80: 
        base_size = 1.25
    else: 
        base_size = 1.50
    
    # ✅ Apply regime-specific caps (explicit)
    if regime in ["R3", "R5"]:
        # SHORT regimes: conservative
        base_size = min(base_size, 1.0)
    elif regime in ["R1", "R2"]:
        # LONG regimes: moderate
        if direction == "LONG":
            base_size = min(base_size, 1.25)
        else:
            base_size = min(base_size, 1.0)
    else:
        base_size = min(base_size, 1.0)
    
    return base_size

# ========== UTILITY FUNCTIONS ==========
def to_utc(ts):
    """Ensure timestamp is UTC"""
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def rolling_zscore_safe(series, window=90):
    """FIXED: Shift AFTER calculation to prevent leakage"""
    return ((series - series.rolling(window).mean()) / 
            series.rolling(window).std()).shift(1)

def rolling_feature_safe(series, window, func='mean'):
    """Safe rolling with shift"""
    if func == 'mean':
        return series.rolling(window).mean().shift(1)
    elif func == 'std':
        return series.rolling(window).std().shift(1)
    elif func == 'median':
        return series.rolling(window).median().shift(1)
    return series

# ========== PRICE NOT NEAR LOWS HELPER ==========
def price_not_near_lows(row, df, lookback=90, min_pct=0.25):
    """
    ✅ FIX 1: Require price to be above X percentile of recent range
    Only applies to LONG positions
    """
    if pd.isna(row['eth_price']):
        return False
    
    idx = row.name
    start_idx = max(0, idx - lookback)
    recent_prices = df.iloc[start_idx:idx]['eth_price'].values
    
    if len(recent_prices) < 10:
        return True  # Not enough data
    
    price_min = np.min(recent_prices)
    price_max = np.max(recent_prices)
    
    if price_max - price_min < 1e-9:
        return True
    
    pct = (row['eth_price'] - price_min) / (price_max - price_min)
    return pct >= min_pct

# ========== PRICE NOT NEAR HIGHS HELPER ==========
def price_not_near_highs(row, df, lookback=90, max_pct=0.75):
    """
    Protection for LONG entries against buying local tops
    Only applies to LONG positions
    """
    if pd.isna(row['eth_price']):
        return True
    
    idx = row.name
    start_idx = max(0, idx - lookback)
    recent_prices = df.iloc[start_idx:idx]['eth_price'].values
    
    if len(recent_prices) < 10:
        return True
    
    price_min = np.min(recent_prices)
    price_max = np.max(recent_prices)
    
    if price_max - price_min < 1e-9:
        return True
    
    pct = (row['eth_price'] - price_min) / (price_max - price_min)
    return pct <= max_pct

# ========== DATA LOADING FROM FILES ==========
def load_data_from_files():
    """
    Load data from files saved by data_loader.py
    Now includes funding data
    """
    print("📂 Loading data from saved files...")
    
    files_to_load = {
        'whale_data': 'data/whale_ml_ready.csv',
        'market_intent': 'data/market_intent_ml_ready.csv', 
        'btc_price': 'data/price_cache/btc.csv',
        'eth_price': 'data/price_cache/eth.csv',
        'funding_data': 'data/funding_rates_ml_ready.csv'
    }
    
    loaded_data = {}
    
    for name, filepath in files_to_load.items():
        if os.path.exists(filepath):
            try:
                if 'price' in name:
                    df = pd.read_csv(filepath, parse_dates=["date"])
                    df["date"] = df["date"].apply(to_utc)
                else:
                    df = pd.read_csv(filepath, parse_dates=["block_date"])
                    df["block_date"] = df["block_date"].apply(to_utc)
                
                loaded_data[name] = df
                print(f"✅ Loaded {name}: {len(df)} rows")
                
                # Special handling for funding data
                if name == 'funding_data' and not df.empty:
                    # Verify funding data covers whale data dates
                    whale_dates = loaded_data.get('whale_data', pd.DataFrame())
                    if not whale_dates.empty:
                        funding_dates = df['block_date']
                        whale_min = whale_dates['block_date'].min()
                        whale_max = whale_dates['block_date'].max()
                        
                        funding_coverage = (
                            funding_dates.min() <= whale_min and
                            funding_dates.max() >= whale_max
                        )
                        
                        if funding_coverage:
                            print(f"   ✅ Full coverage: {whale_min.date()} to {whale_max.date()}")
                        else:
                            print(f"   ⚠️ Partial coverage")
                            print(f"   Whale: {whale_min.date()} to {whale_max.date()}")
                            print(f"   Funding: {funding_dates.min().date()} to {funding_dates.max().date()}")
                
            except Exception as e:
                print(f"❌ Failed to load {name}: {e}")
                loaded_data[name] = pd.DataFrame()
        else:
            print(f"❌ {name} file not found: {filepath}")
            if name == 'funding_data':
                print(f"   ⚠️ Funding data missing - system will use zeros")
            loaded_data[name] = pd.DataFrame()
    
    # Check essential data
    essential_data = ['whale_data', 'market_intent', 'btc_price', 'eth_price']
    if all(len(loaded_data[d]) > 0 for d in essential_data):
        print(f"\n✅ Essential data loaded successfully")
    else:
        print(f"\n⚠️  Some essential data files are missing or empty")
        print(f"   Please run data_loader.py to fetch fresh data")
    
    return (
        loaded_data.get('whale_data', pd.DataFrame()),
        loaded_data.get('market_intent', pd.DataFrame()),
        loaded_data.get('btc_price', pd.DataFrame()),
        loaded_data.get('eth_price', pd.DataFrame()),
        loaded_data.get('funding_data', pd.DataFrame())
    )

# ========== FEATURE ENGINEERING ==========
def add_price_features(df, price_col, prefix):
    """Add technical features for a price series"""
    df = df.copy()
    
    # Log returns
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    # Lagged returns (including lag 2 for LONG confirmation)
    for lag in [1, 2, 3, 7]:
        df[f'{prefix}_ret_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    # Volatility
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7).std().shift(1)
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30).std().shift(1)
    
    # RSI
    returns = df[f'{prefix}_log_return']
    gains = returns.where(returns > 0, 0).rolling(14).mean()
    losses = -returns.where(returns < 0, 0).rolling(14).mean()
    df[f'{prefix}_rsi'] = (100 - (100 / (1 + gains / (losses + 1e-10)))).shift(1)
    
    return df

def engineer_features(df_whales, df_market_intent, df_btc, df_eth, df_funding=None):
    """Engineer all features with funding data from loader"""
    print("🔧 Engineering features...")
    
    # Merge price data
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    
    # Merge with whale data
    df = pd.merge(
        df_whales, 
        df_prices, 
        left_on='block_date', 
        right_on='date', 
        how='left'
    ).drop(columns=['date'])
    
    # Merge with market intent data
    df = pd.merge(
        df, 
        df_market_intent, 
        on='block_date', 
        how='left', 
        suffixes=('', '_intent')
    )
    
    # ========== MERGE FUNDING DATA ==========
    if df_funding is not None and not df_funding.empty:
        # Merge but keep as separate column
        df = pd.merge(
            df,
            df_funding[['block_date', 'eth_funding_rate_8h']],
            on='block_date',
            how='left'  # LEFT join - keep all dates even if no funding
        )
        print(f"✅ Merged funding data: {len(df_funding)} rows")
        
        # Check what percentage has funding data
        funding_present = df['eth_funding_rate_8h'].notna().sum()
        funding_pct = funding_present / len(df) * 100
        print(f"   Funding coverage: {funding_present}/{len(df)} rows ({funding_pct:.1f}%)")
    else:
        # Don't create the column if no data
        print("⚠️ No funding data - funding column will not be created")
    
    
    df = df.sort_values('block_date').reset_index(drop=True)
    
    # Add price features
    df = add_price_features(df, 'eth_price', 'eth')
    df = add_price_features(df, 'btc_price', 'btc')
    
    # ETH/BTC ratio features
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7).mean().shift(1)
    df['eth_btc_corr_30d'] = df['eth_log_return'].shift(1).rolling(30) \
        .corr(df['btc_log_return'].shift(1)).shift(1)
        
    # ========== ADD VOL_RATIO FEATURE ==========
    if 'eth_vol7' in df.columns and 'eth_vol30' in df.columns:
        df['vol_ratio'] = df['eth_vol7'] / df['eth_vol30']
        df['vol_ratio'] = df['vol_ratio'].clip(0.5, 2.0)
        print("✅ Created vol_ratio feature")
    else:
        print("⚠️  Could not create vol_ratio - missing eth_vol7 or eth_vol30")
        df['vol_ratio'] = 1.0
            
    # Apply safe rolling z-scores
    zscore_pairs = [
        ('whale_tx_count', 'whale_tx_zscore_90d'),
        ('tx_per_active', 'tx_per_active_zscore_90d'),
        ('eth_burned', 'eth_burned_zscore_90d'),
        ('exchange_volume', 'exchange_volume_zscore'),
    ]
    
    for raw_col, zscore_col in zscore_pairs:
        if raw_col in df.columns:
            df[zscore_col] = rolling_zscore_safe(df[raw_col], 90)
    
    # Burn/issuance ratio
    if all(col in df.columns for col in ['eth_burned', 'total_gas_fees']):
        df['burn_issuance_ratio'] = (df['eth_burned'] / (df['total_gas_fees'] + 1e-10)).shift(1)
    
    # Whale volume deltas
    if 'whale_volume_ratio' in df.columns:
        df['whale_volume_ratio_delta_1d'] = df['whale_volume_ratio'].diff(1).shift(1)
        df['whale_volume_ratio_delta_3d'] = df['whale_volume_ratio'].diff(3).shift(1)
    
    # Ensure ALL LONG_FEATURES exist
    for feature in LONG_FEATURES:
        if feature not in df.columns:
            print(f"⚠️  Creating missing LONG feature: {feature}")
            df[feature] = 0.0
    
    # Ensure ALL SHORT_FEATURES exist  
    for feature in SHORT_FEATURES:
        if feature not in df.columns:
            print(f"⚠️  Creating missing SHORT feature: {feature}")
            df[feature] = 0.0
    
    # Clean up intermediate columns
    df = df.drop(columns=['eth_log_return', 'btc_log_return'], errors='ignore')
    
    # Save engineered features
    df.to_csv('data/features_engineered.csv', index=False)
    print(f"✅ Features engineered: {len(df.columns)} columns, {len(df)} rows")
    print(f"   LONG features available: {sum(1 for f in LONG_FEATURES if f in df.columns)}/{len(LONG_FEATURES)}")
    print(f"   SHORT features available: {sum(1 for f in SHORT_FEATURES if f in df.columns)}/{len(SHORT_FEATURES)}")
    
    return df

# ========== TARGET CREATION ==========
def create_targets_two_tier(df, k=1.5):
    """
    Create two-tier SHORT labels (crash + breakdown)
    """
    print("🎯 Creating two-tier targets...")
    
    df = df.sort_values('block_date').reset_index(drop=True).copy()
    
    # Calculate returns
    df['eth_log_return'] = np.log(df['eth_price'] / df['eth_price'].shift(1))
    df['rolling_vol_30'] = df['eth_log_return'].rolling(30, min_periods=10).std()
    
    # T+2 returns and threshold
    df['return_t2'] = df['eth_log_return'].rolling(2).sum().shift(-2)
    
    # Dynamic threshold using 65th percentile
    df['threshold_t2'] = df['rolling_vol_30'].rolling(60, min_periods=20).quantile(0.65)
    
    # Tier 1: Crash (hard down)
    hard_down = (
        (df['return_t2'] < -df['threshold_t2']) &
        (df['eth_vol7'] > df['eth_vol30']).fillna(False)
    )
    
    # Tier 2: Breakdown (pre-crash)
    exchange_flow_median = df['exchange_flow_share'].rolling(90, min_periods=30).median()
    
    soft_down = (
        (df['eth_ret_lag1'].fillna(0) < 0) &
        (df['btc_ret_lag1'].fillna(0) < 0) &
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Create targets
    df['target_t2'] = 0
    df.loc[df['return_t2'] > df['threshold_t2'], 'target_t2'] = 1  # UP
    df.loc[hard_down | soft_down, 'target_t2'] = -1  # DOWN (both tiers)
    
    # Create binary targets
    df['y_long_t2'] = (df['target_t2'] == 1).astype(int)
    df['y_short_t2'] = (df['target_t2'] == -1).astype(int)
    
    # Clean up
    df = df.drop(columns=['eth_log_return'], errors='ignore')
    
    # Print distribution
    print("\n📊 Target Distribution (Two-Tier SHORT):")
    for state, label in [(-1, 'DOWN'), (0, 'FLAT'), (1, 'UP')]:
        count = (df['target_t2'] == state).sum()
        percentage = count / len(df) * 100
        print(f"  {label:5s}: {count:4d} ({percentage:5.1f}%)")
    
    hard_count = hard_down.sum()
    soft_count = soft_down.sum()
    total_down = (df['target_t2'] == -1).sum()
    
    print(f"\n  Tier 1 (crash):     {hard_count:4d}")
    print(f"  Tier 2 (breakdown): {soft_count:4d}")
    print(f"  Total DOWN:         {total_down:4d}")
    
    return df

# ========== REGIME DEFINITION ==========
def define_regimes_extended(df):
    """Define trading regimes including R5 distribution regime"""
    print("📈 Defining extended regimes...")
    
    if 'btc_ret_lag1' not in df.columns or 'eth_vol7' not in df.columns:
        df['regime_code'] = 'R0'
        return df
    
    # Standard regimes based on BTC trend and ETH volatility
    btc_trend_7d = df['btc_ret_lag1'].rolling(7).mean()
    df['btc_regime'] = pd.cut(
        btc_trend_7d, 
        bins=[-np.inf, -0.005, 0.005, np.inf], 
        labels=['DOWN', 'FLAT', 'UP']
    )
    
    vol_median = df['eth_vol7'].rolling(180, min_periods=60).median()
    df['vol_regime'] = (df['eth_vol7'] > vol_median).map({True: 'HIGH', False: 'LOW'})
    
    # Combine for standard regimes
    df['regime'] = df['btc_regime'].astype(str) + '_' + df['vol_regime'].astype(str)
    regime_map = {
        'UP_HIGH': 'R1',    # Bull high vol
        'UP_LOW': 'R2',     # Bull low vol
        'DOWN_HIGH': 'R3',  # Bear high vol
        'DOWN_LOW': 'R4',   # Bear low vol
    }
    df['regime_code'] = df['regime'].map(regime_map).fillna('R0')
    
    # R5: Whale distribution regime
    exchange_flow_median = df['exchange_flow_share'].rolling(60, min_periods=20).median()
    
    df['dist_regime'] = (
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Override with R5 where distribution regime is active
    df.loc[df['dist_regime'], 'regime_code'] = 'R5'
    
    # Print regime distribution
    print("\n📊 Extended Regime Distribution:")
    regime_stats = []
    for code in ['R1', 'R2', 'R3', 'R4', 'R5', 'R0']:
        count = (df['regime_code'] == code).sum()
        if len(df) > 0:
            pct = count / len(df) * 100
            icon = '🟢' if code == 'R1' else ('🔴' if code in ['R3', 'R5'] else '⚪')
            regime_stats.append(f"{icon} {code}: {count:4d} ({pct:5.1f}%)")
    
    # Print in two columns
    for i in range(0, len(regime_stats), 2):
        row = regime_stats[i:i+2]
        print("  " + " | ".join(row))
    
    return df

# ========== BUILD COMPLETE PIPELINE ==========
def build_pipeline_complete(df_features):
    """
    Create the complete pipeline dataset with features, targets, and regimes
    """
    print("\n" + "="*70)
    print("BUILDING COMPLETE PIPELINE DATASET")
    print("="*70)
    
    # Create targets
    df_with_targets = create_targets_two_tier(df_features)
    
    # Define regimes
    df_complete = define_regimes_extended(df_with_targets)
    
    # Ensure all required features exist
    for feature in LONG_FEATURES + SHORT_FEATURES:
        if feature not in df_complete.columns:
            df_complete[feature] = 0.0
    
    # Fill NaN values for features
    feature_cols = [col for col in df_complete.columns if col not in 
                   ['block_date', 'target_t2', 'y_long_t2', 'y_short_t2', 
                    'regime_code', 'btc_regime', 'vol_regime', 'regime', 'dist_regime']]
    
    df_complete[feature_cols] = df_complete[feature_cols].fillna(method='ffill').fillna(0)
    
    # Save complete pipeline
    df_complete.to_csv('data/pipeline_complete.csv', index=False)
    
    # Report statistics
    print(f"\n✅ Pipeline complete saved:")
    print(f"   Rows: {len(df_complete)}")
    print(f"   Columns: {len(df_complete.columns)}")
    print(f"   Date range: {df_complete['block_date'].min().date()} to {df_complete['block_date'].max().date()}")
    print(f"   File: data/pipeline_complete.csv")
    
    # Feature availability report
    print(f"\n📊 Feature Availability:")
    print(f"   LONG features: {sum(1 for f in LONG_FEATURES if f in df_complete.columns)}/{len(LONG_FEATURES)}")
    print(f"   SHORT features: {sum(1 for f in SHORT_FEATURES if f in df_complete.columns)}/{len(SHORT_FEATURES)}")
    
    # Check for missing features
    missing_long = [f for f in LONG_FEATURES if f not in df_complete.columns]
    missing_short = [f for f in SHORT_FEATURES if f not in df_complete.columns]
    
    if missing_long:
        print(f"   ⚠️  Missing LONG features: {missing_long}")
    if missing_short:
        print(f"   ⚠️  Missing SHORT features: {missing_short}")
    
    return df_complete

# ========== SHORT-SPECIFIC LOGIC ==========
def check_r3_short_allowed(row):
    """
    R3 short philosophy (early weakness only)
    """
    # Small red, not dump
    small_red = (-0.015 < row.get('eth_ret_lag1', 0) < 0)
    
    # BTC weakening
    btc_weak = (row.get('btc_ret_lag3', 0) < 0)
    
    # NOT vol expansion (early, not panic)
    vol_ratio = row.get('vol_ratio', 1)
    no_vol_expansion = (vol_ratio <= 1.0)
    
    # Whales increasing activity
    whale_activity = (row.get('whale_volume_ratio_delta_3d', 0) > 0)
    
    return small_red and btc_weak and no_vol_expansion and whale_activity

def calculate_short_veto_score(row):
    """Calculate veto scores for SHORT positions"""
    veto = 0
    reasons = []
    
    structural_score = 0
    flow_score = 0
    context_score = 0
    
    # Flow vetoes
    if row.get('net_exchange_flow_ratio', 0) < 0 and row.get('exchange_volume_zscore', 0) > 0:
        veto += 1
        flow_score += 1
        reasons.append('net_flow_negative_with_liquidity')
    
    if row.get('whale_exchange_flow_ratio', 0) > 0.6:
        veto += 1
        flow_score += 1
        reasons.append('whale_to_exchange')
    
    # Structural vetoes
    if row.get('btc_ret_lag1', 0) < -0.02 and row.get('eth_ret_lag1', 0) < -0.01:
        veto += 2
        structural_score += 2
        reasons.append('btc_breakdown')
    
    # Structural vetoes - UPDATED vol expansion check
    if row.get('vol_ratio', 1) > 1.0:  # CHANGED: eth_vol7 > eth_vol30 → vol_ratio > 1.0
        veto += 2
        structural_score += 2
        reasons.append('vol_expansion')
    
    # Context vetoes - UPDATED low volatility check
    if row.get('vol_ratio', 1) < 0.7:  # CHANGED: eth_vol7 < eth_vol30*0.7 → vol_ratio < 0.7
        veto += 1
        context_score += 1
        reasons.append('low_volatility')
    
    return veto, structural_score, flow_score, context_score, reasons
        
# ========== FUNDING CONFIDENCE MODIFIER ==========
def apply_funding_modifier(row, direction, current_confidence):
    """
    Apply funding rate as OPTIONAL confidence modifier
    Only when funding data exists (post-2025)
    """
    # Get funding rate - returns None if column doesn't exist or is NaN
    if 'eth_funding_rate_8h' not in row.index:
        return current_confidence, []  # No funding column at all
    
    funding = row.get('eth_funding_rate_8h')
    
    # CRITICAL: If funding is None or NaN, do NOTHING
    if funding is None or pd.isna(funding):
        return current_confidence, []
    
    reasons = []
    funding_adjustment = 0.0
    
    # Asymmetric thresholds (same as before but as modifier, not veto)
    FUNDING_EUPHORIA_LEVEL = 0.0003  # 0.03% for crowded longs
    FUNDING_PANIC_LEVEL = -0.0002    # -0.02% for crowded shorts
    
    if direction == "LONG":
        if funding > FUNDING_EUPHORIA_LEVEL:
            # Crowded longs reduce confidence
            funding_adjustment = -0.03
            reasons.append(f"crowded_longs_funding_{funding*100:.3f}%")
    
    elif direction == "SHORT":
        if funding < FUNDING_PANIC_LEVEL:
            # Crowded shorts reduce confidence
            funding_adjustment = -0.03
            reasons.append(f"crowded_shorts_funding_{funding*100:.3f}%")
        elif funding > FUNDING_EUPHORIA_LEVEL:
            # High positive funding boosts SHORT confidence
            funding_adjustment = +0.03
            reasons.append(f"euphoric_funding_boost_{funding*100:.3f}%")
    
    # Apply adjustment with bounds
    adjusted_confidence = current_confidence + funding_adjustment
    adjusted_confidence = max(0.0, min(1.0, adjusted_confidence))
    
    return adjusted_confidence, reasons

    
def check_short_requirements(row, regime, structural_score, flow_score):
    """Check SHORT-specific requirements with nuanced flow confirmation"""
    reasons = []
    
    # Flow confirmation with nuance
    if flow_score == 0:
        # Allow only if structural weakness + bearish BTC context
        if not (
            structural_score > 0 and
            row.get('btc_ret_lag1', 0) < 0
        ):
            reasons.append("no_flow_confirmation")
    
    # R5 stronger flow requirement (only if we have flow signals)
    if regime == "R5" and flow_score > 0 and flow_score < 2:
        reasons.append("weak_distribution_flow")
    
    # Structural check - no structural weakness = no short
    if structural_score == 0:
        reasons.append("no_structural_break")
    
    # R3 short check (using new philosophy)
    if regime == "R3" and not check_r3_short_allowed(row):
        reasons.append("r3_no_early_weakness")
    
    return reasons

# ========== LONG-SPECIFIC LOGIC ==========
def long_veto(row):
    """LONG veto - minimal and asymmetric"""
    veto = []
    
    if row.get('btc_ret_lag1', 0) < -0.02:
        veto.append("btc_drawdown")
    
    if row.get('whale_exchange_flow_ratio', 0) > 0.6:
        veto.append("distribution")
    
    if row.get('eth_vol7', 0) > row.get('eth_vol30', 0) * 1.5:
        veto.append("vol_spike")
    
    return veto

def long_confirmation(row):
    """
    LONG confirmation logic
    Stage A: ML finds accumulation
    Stage B: Confirm price is responding
    """
    # Updated to use vol_ratio
    vol_ratio = row.get('vol_ratio', 1)
    
    return (
        row.get('eth_ret_lag1', 0) > 0 and
        row.get('eth_ret_lag2', 0) > 0 and
        vol_ratio < 1.0  # Volatility compression (accumulation)
    )
# ========== MODEL MANAGEMENT ==========

def rebuild_models_if_needed(df_pipeline):
    """
    Rebuild models if they don't exist or feature mismatch
    Returns: (short_model, long_model)
    """
    short_model = None
    long_model = None
    
    # Check SHORT model
    short_model_path = 'models/r5_short_final.pkl'
    if os.path.exists(short_model_path):
        try:
            short_model = joblib.load(short_model_path)
            print(f"✅ SHORT model loaded from {short_model_path}")
            
            # Check if model has feature_names_ attribute
            if hasattr(short_model, 'feature_names_'):
                print(f"   Model expects {len(short_model.feature_names_)} features")
                missing = [f for f in short_model.feature_names_ if f not in df_pipeline.columns]
                if missing:
                    print(f"⚠️  SHORT model missing features: {missing[:5]}")
                    print("   Rebuilding SHORT model...")
                    short_model = None
            else:
                print("⚠️  SHORT model missing feature_names_, rebuilding...")
                short_model = None
        except Exception as e:
            print(f"⚠️  Error loading SHORT model: {e}")
            short_model = None
    
    # Check LONG model
    long_model_path = 'models/R1_R2_LONG.pkl'
    if os.path.exists(long_model_path):
        try:
            long_model = joblib.load(long_model_path)
            print(f"✅ LONG model loaded from {long_model_path}")
            
            # Check if model has feature_names_ attribute
            if hasattr(long_model, 'feature_names_'):
                print(f"   Model expects {len(long_model.feature_names_)} features")
                missing = [f for f in long_model.feature_names_ if f not in df_pipeline.columns]
                if missing:
                    print(f"⚠️  LONG model missing features: {missing[:5]}")
                    print("   Rebuilding LONG model...")
                    long_model = None
            else:
                print("⚠️  LONG model missing feature_names_, rebuilding...")
                long_model = None
        except Exception as e:
            print(f"⚠️  Error loading LONG model: {e}")
            long_model = None
    
    # Rebuild SHORT model if needed
    if short_model is None:
        print("\n" + "="*70)
        print("REBUILDING SHORT MODEL (R5)")
        print("="*70)
        
        df_r5 = df_pipeline[df_pipeline['regime_code'] == 'R5'].copy()
        
        # ✅ CRITICAL FIX 1: Use only features that exist in the data
        short_features = [f for f in SHORT_FEATURES if f in df_r5.columns]
        print(f"   Using {len(short_features)} SHORT features: {short_features}")
        
        if len(df_r5) >= 50:
            split_idx = int(len(df_r5) * 0.8)
            X_train_short = df_r5[short_features].iloc[:split_idx].fillna(0)
            y_train_short = df_r5['y_short_t2'].iloc[:split_idx]
            
            print(f"   Training on {len(X_train_short)} R5 samples")
            
            short_model = GradientBoostingClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42
            )
            short_model.fit(X_train_short, y_train_short)
            
            # ✅ CRITICAL FIX 2: Store feature names in the model
            short_model.feature_names_ = short_features
            joblib.dump(short_model, short_model_path)
            
            # Verify
            print(f"✅ SHORT model rebuilt and saved")
            print(f"   Model now expects {len(short_model.feature_names_)} features")
            print(f"   Features: {short_model.feature_names_}")
        else:
            print("⚠️  Insufficient R5 data for SHORT model")
    
    # Rebuild LONG model if needed
    if long_model is None:
        print("\n" + "="*70)
        print("REBUILDING LONG MODEL (R1+R2)")
        print("="*70)
        
        df_long = df_pipeline[df_pipeline['regime_code'].isin(['R1', 'R2'])].copy()
        
        # Remove obvious traps
        df_long = df_long[
            (df_long['btc_ret_lag1'] > -0.02) &
            (df_long['whale_exchange_flow_ratio'] < 0.6)
        ]
        
        # ✅ CRITICAL FIX 3: Use only features that exist in the data
        long_features = [f for f in LONG_FEATURES if f in df_long.columns]
        print(f"   Using {len(long_features)} LONG features: {long_features}")
        
        X_long = df_long[long_features].fillna(0)
        y_long = df_long['y_long_t2']
        
        if len(X_long) >= 50:
            long_model = GradientBoostingClassifier(
                n_estimators=120,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                random_state=42
            )
            
            long_model.fit(X_long, y_long)
            
            # ✅ CRITICAL FIX 4: Store feature names in the model
            long_model.feature_names_ = long_features
            joblib.dump(long_model, long_model_path)
            
            print("✅ LONG model rebuilt and saved")
            print(f"   Model now expects {len(long_model.feature_names_)} features")
            print(f"   Features: {long_model.feature_names_}")
            
            # Basic validation
            probs = long_model.predict_proba(X_long)[:, 1]
            preds = (probs >= 0.60).astype(int)
            prec = precision_score(y_long, preds, zero_division=0)
            rec = recall_score(y_long, preds, zero_division=0)
            
            print(f"   Training precision: {prec:.3f}")
            print(f"   Training recall: {rec:.3f}")
        else:
            print("⚠️  Insufficient LONG data")
    
    return short_model, long_model

# ========== UNIFIED SIGNAL GENERATION ==========

def generate_unified_signal(row, df, short_model, long_model):
    """
    UNIFIED SIGNAL GENERATION with optional funding modifier
    """
    regime = row.get('regime_code', 'R0')
    date_str = str(row['block_date'].date()) if 'block_date' in row else str(row.name)
    
    # Base signal object
    signal = {
        "date": date_str,
        "regime": regime,
        "direction": None,
        "model_probability": 0.0,
        "adjusted_confidence": 0.0,
        "position_size": 0.0,
        "reasons": [],
        "funding_available": False,
        "funding_rate": None,
        "action": "NO_TRADE"
    }
    
    # Record if funding data exists for this row
    if 'eth_funding_rate_8h' in row.index and not pd.isna(row.get('eth_funding_rate_8h')):
        signal["funding_available"] = True
        signal["funding_rate"] = float(row['eth_funding_rate_8h'])
    
    # ===== LONG LOGIC (R1/R2/R3) =====
    if regime in ['R1', 'R2', 'R3'] and long_model:
        signal["direction"] = "LONG"
        
        # Use model's stored feature names for inference
        if not hasattr(long_model, 'feature_names_'):
            signal["reasons"] = ["model_error: long_model missing feature_names_"]
            return signal
        
        features = long_model.feature_names_
        try:
            # Ensure features exist in row and fill missing with 0
            X = row.reindex(features, fill_value=0).values.reshape(1, -1)
            prob = long_model.predict_proba(X)[0, 1]
            signal["model_probability"] = float(prob)
        except Exception as e:
            signal["reasons"] = [f"model_error: {str(e)[:100]}"]
            return signal
        
        # Step 1: Check probability threshold
        if prob < LONG_ENTRY_THRESHOLD:
            signal["reasons"] = ["low_model_probability"]
            return signal
        
        # Step 2: Calculate confirmation score
        confirm_score = 0

        if row.get('eth_ret_lag1', 0) > 0:
            confirm_score += 1
        if row.get('eth_ret_lag2', 0) > 0:
            confirm_score += 1
        if row.get('vol_ratio', 1) < 1.0:
            confirm_score += 1
        
        # ✅ FIX 1: Define required_score based on regime
        if regime == "R1":
            required_score = 1  # Early bull: allow early signs
        elif regime == "R2":
            required_score = 2  # Late bull: require agreement
        elif regime == "R3":
            required_score = 3  # Early bear: extremely strict
        else:
            required_score = 2  # Safe default
        
        if confirm_score < required_score:
            signal["reasons"] = [f"weak_price_confirmation ({confirm_score}/{required_score})"]
            return signal
        
        # Step 3: Apply LONG vetoes
        veto_reasons = long_veto(row)
        if veto_reasons:
            signal["reasons"] = veto_reasons
            return signal
        
        # Step 4: Calculate confidence WITH OPTIONAL FUNDING MODIFIER
        veto_score = len(veto_reasons)
        adj_conf, funding_reasons = adjust_confidence_unified(
            prob, regime, direction="LONG", veto_score=veto_score, row=row
        )
        signal["adjusted_confidence"] = adj_conf
        
        # Add funding reasons to signal if any
        if funding_reasons:
            signal["reasons"].extend(funding_reasons)
        
        # Step 5: Regime-aware confidence floor
        if regime == "R1":
            confidence_floor = 0.50  # Lower for early bull
        elif regime == "R2":
            confidence_floor = 0.55  # Higher for late bull
        elif regime == "R3":
            confidence_floor = 0.60  # Highest for early bear (should be rare)
        else:
            confidence_floor = 0.55  # Default
        
        if signal["adjusted_confidence"] < confidence_floor:
            signal["reasons"] = ["low_final_confidence"]
            signal["direction"] = None
            return signal
        
        # ✅ FIXED: Use the updated position sizing function
        signal["position_size"] = map_confidence_to_size_unified(
            signal["adjusted_confidence"], regime, direction="LONG"
        )
        signal["reasons"] = ["ml_accumulation", "price_confirmation"]
        signal["action"] = "ENTER"
        return signal
        
    # ===== SHORT LOGIC (R3/R5) =====
          
    elif regime in ['R3', 'R5'] and short_model:
        signal["direction"] = "SHORT"
        
        # Use model's stored feature names for inference
        if not hasattr(short_model, 'feature_names_'):
            signal["reasons"] = ["model_error: short_model missing feature_names_"]
            return signal
        
        features = short_model.feature_names_
        try:
            # Ensure features exist in row and fill missing with 0
            X = row.reindex(features, fill_value=0).values.reshape(1, -1)
            prob = short_model.predict_proba(X)[0, 1]
            signal["model_probability"] = float(prob)
        except Exception as e:
            signal["reasons"] = [f"model_error: {str(e)[:100]}"]
            return signal
        
        # Step 1: Check probability threshold (HIGHER for SHORT)
        if prob < SHORT_ENTRY_THRESHOLD:
            signal["reasons"] = ["low_model_probability"]
            return signal
        
        # Step 2: Calculate veto scores
        veto, structural_score, flow_score, context_score, veto_reasons = calculate_short_veto_score(row)
        
        # Step 3: Check SHORT-specific requirements
        requirement_failures = check_short_requirements(row, regime, structural_score, flow_score)
        if requirement_failures:
            signal["reasons"] = requirement_failures
            return signal
        
        # Step 4: Calculate confidence WITH OPTIONAL FUNDING MODIFIER
        adj_conf, funding_reasons = adjust_confidence_unified(
            prob, regime, direction="SHORT", veto_score=veto, row=row
        )
        signal["adjusted_confidence"] = adj_conf
        
        # Add funding reasons if any
        if funding_reasons:
            signal["reasons"].extend(funding_reasons)
        
        # Final confidence check
        if signal["adjusted_confidence"] < 0.55:
            signal["reasons"] = ["low_final_confidence"]
            signal["direction"] = None
            return signal
        
        signal["position_size"] = map_confidence_to_size_unified(
            signal["adjusted_confidence"], regime, direction="SHORT"
        )
        signal["reasons"] = veto_reasons  # Use veto reasons as trade reasons
        signal["action"] = "ENTER"
        return signal
    
    # ===== NEUTRAL REGIME =====
    else:
        signal["reasons"] = ["neutral_regime"]
        return signal
            
def generate_daily_signal_unified(df, short_model, long_model):
    """
    Generate unified daily signal (uses latest row)
    """
    latest_row = df.iloc[-1].copy()
    return generate_unified_signal(latest_row, df, short_model, long_model)

# ========== SIGNAL INSPECTION ==========
def inspect_signals_unified(df, short_model, long_model, num_signals=60):
    """
    Inspect signals with optional funding modifier
    """
    print("\n" + "="*70)
    print(f"UNIFIED SIGNAL INSPECTION (Last {num_signals} days)")
    print("="*70)
    
    # Count funding availability
    funding_available = 0
    if 'eth_funding_rate_8h' in df.columns:
        recent = df.iloc[-num_signals:]
        funding_available = recent['eth_funding_rate_8h'].notna().sum()
        print(f"Funding data available: {funding_available}/{num_signals} days ({funding_available/num_signals*100:.1f}%)")
    
    recent_data = df.iloc[-num_signals:].copy()
    print(f"Date range: {recent_data['block_date'].min().date()} to {recent_data['block_date'].max().date()}")
    
    signals = []
    long_count = 0
    short_count = 0
    no_trade_count = 0
    
    for idx, row in recent_data.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
        
        # Print only trade signals
        if signal['action'] == 'ENTER':
            if signal['direction'] == 'LONG':
                long_count += 1
                funding_status = f" | Funding: {signal.get('funding_rate', 0)*100:.3f}%" if signal.get('funding_rate') is not None else ""
                print(f"\n🟢 LONG:  {signal['date']} | Regime: {signal['regime']} | "
                      f"Conf: {signal['adjusted_confidence']:.2f} | Size: {signal['position_size']:.2f}{funding_status}")
            elif signal['direction'] == 'SHORT':
                short_count += 1
                funding_status = f" | Funding: {signal.get('funding_rate', 0)*100:.3f}%" if signal.get('funding_rate') is not None else ""
                print(f"\n🔴 SHORT: {signal['date']} | Regime: {signal['regime']} | "
                      f"Conf: {signal['adjusted_confidence']:.2f} | Size: {signal['position_size']:.2f}{funding_status}")
        else:
            no_trade_count += 1
    
    print(f"\n📊 Signal Summary:")
    print(f"   Total days: {len(signals)}")
    print(f"   LONG signals: {long_count} ({long_count/len(signals)*100:.1f}%)")
    print(f"   SHORT signals: {short_count} ({short_count/len(signals)*100:.1f}%)")
    print(f"   NO_TRADE: {no_trade_count} ({no_trade_count/len(signals)*100:.1f}%)")
    
      
    # Regime distribution
    print(f"\n📈 Regime Distribution:")
    regime_dist = {}
    for signal in signals:
        regime = signal.get('regime', 'UNKNOWN')
        regime_dist[regime] = regime_dist.get(regime, 0) + 1
    
    for regime in sorted(regime_dist.keys()):
        count = regime_dist[regime]
        print(f"   {regime}: {count} days ({count/len(signals)*100:.1f}%)")
    
    # Rejection reasons analysis
    print(f"\n🔍 Rejection Reasons:")
    rejection_reasons = {}
    for signal in signals:
        if signal['action'] == 'NO_TRADE' and signal.get('reasons'):
            for reason in signal['reasons']:
                rejection_reasons[reason] = rejection_reasons.get(reason, 0) + 1
    
    for reason, count in sorted(rejection_reasons.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"   {reason}: {count} times")
    
    return signals
def inspect_long_signals_bull_cycle(df, short_model, long_model, start_date='2020-01-01', end_date='2022-01-01'):
    """
    Inspect LONG signals during the 2020-2021 bull cycle
    """
    print("\n" + "="*70)
    print(f"LONG SIGNAL INSPECTION: {start_date} to {end_date}")
    print("="*70)
    
    # Filter to bull cycle period
    mask = (df['block_date'] >= start_date) & (df['block_date'] <= end_date)
    bull_data = df[mask].copy()
    
    print(f"Period: {bull_data['block_date'].min().date()} to {bull_data['block_date'].max().date()}")
    print(f"Total days: {len(bull_data)}")
    
    # Get R1/R2 days
    bull_regimes = bull_data[bull_data['regime_code'].isin(['R1', 'R2'])].copy()
    print(f"R1/R2 days: {len(bull_regimes)}")
    print(f"  R1: {(bull_regimes['regime_code'] == 'R1').sum()} days")
    print(f"  R2: {(bull_regimes['regime_code'] == 'R2').sum()} days")
    
    # Generate signals for R1/R2 days only
    long_signals = []
    for idx, row in bull_regimes.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        if signal['direction'] == 'LONG':
            long_signals.append(signal)
    
    # Analyze LONG signals
    print(f"\n📊 LONG Signal Analysis:")
    print(f"  Total LONG signals: {len(long_signals)}")
    
    if len(long_signals) > 0:
        df_long_signals = pd.DataFrame(long_signals)
        
        # Group by regime
        regime_counts = df_long_signals['regime'].value_counts()
        for regime, count in regime_counts.items():
            print(f"  {regime}: {count} signals")
        
        # Analyze timing (early vs late bull)
        df_long_signals['date_dt'] = pd.to_datetime(df_long_signals['date'])
        df_long_signals['month'] = df_long_signals['date_dt'].dt.to_period('M')
        monthly_counts = df_long_signals['month'].value_counts().sort_index()
        
        print(f"\n📅 Monthly distribution:")
        for month, count in monthly_counts.head(12).items():  # Show first 12 months
            print(f"  {month}: {count} signals")
        
        # Check if signals avoid tops
        print(f"\n🔍 Top avoidance check:")
        for signal in df_long_signals.head(5).to_dict('records'):  # Show first 5 signals
            date_str = signal['date']
            row = bull_data[bull_data['block_date'] == pd.Timestamp(date_str)]
            if not row.empty:
                row = row.iloc[0]
                # Check if price near highs (should be False for good LONGs)
                near_highs = not price_not_near_highs(row, df, lookback=90, max_pct=0.75)
                status = "⚠️ NEAR HIGHS" if near_highs else "✅ NOT NEAR HIGHS"
                print(f"  {date_str}: {status} | Price: ${row['eth_price']:.0f} | Conf: {signal['adjusted_confidence']:.2f}")
    
    # Also check how many R1/R2 days were rejected and why
    print(f"\n🔍 LONG Rejection Analysis (R1/R2 days):")
    rejection_counts = {}
    for idx, row in bull_regimes.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        if signal['action'] != 'ENTER' and signal['direction'] == 'LONG':
            for reason in signal['reasons']:
                rejection_counts[reason] = rejection_counts.get(reason, 0) + 1
    
    for reason, count in sorted(rejection_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {reason}: {count} times")
    
    return long_signals

# ========== MAIN PIPELINE ==========
def run_unified_pipeline():
    """
    Execute complete pipeline with funding data from loader
    """
    print("\n" + "="*70)
    print("ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT")
    print("="*70)
    print("✅ Features FROZEN")
    print("✅ Regimes FROZEN")
    print("✅ Funding: Veto-only from data loader")
    print("✅ Volatility: vol_ratio feature")
    print("="*70)
    
    # Step 1: Load all data including funding
    df_whales, df_market, df_btc, df_eth, df_funding = load_data_from_files()
    
    # Check essential data
    if any(d.empty for d in [df_whales, df_market, df_btc, df_eth]):
        print("\n❌ Missing essential data. Run data_loader.py first.")
        return None, None, None
    
    # Step 2: Engineer features with funding data
    df_features = engineer_features(df_whales, df_market, df_btc, df_eth, df_funding)
    
    # Step 3: Build pipeline
    df_pipeline = build_pipeline_complete(df_features)    
    # Step 4: Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df_pipeline)
    
    # Step 5: Generate live signal
    if short_model and long_model:
        print("\n" + "="*70)
        print("GENERATING UNIFIED LIVE SIGNAL")
        print("="*70)
        
        signal = generate_daily_signal_unified(df_pipeline, short_model, long_model)
        print(json.dumps(signal, indent=2))
        
        with open('data/latest_signal_unified.json', 'w') as f:
            json.dump(signal, f, indent=2)
    
    # Step 6: Inspect 60-day history
    if short_model and long_model:
        print("\n" + "="*70)
        print("60-DAY UNIFIED SIGNAL INSPECTION")
        print("="*70)
        
        signals = inspect_signals_unified(df_pipeline, short_model, long_model, 60)
        
        df_signals = pd.DataFrame(signals)
        df_signals.to_csv('validation/signal_inspection_unified.csv', index=False)
        print(f"\n✅ Saved: validation/signal_inspection_unified.csv")
    
    return df_pipeline, short_model, long_model

# ========== PAPER TRADE TEST ==========
def run_unified_paper_test():
    """
    Run 60-day paper trade test with UNIFIED logic
    """
    print("\n" + "="*70)
    print("60-DAY PAPER TRADE TEST (UNIFIED LOGIC)")
    print("="*70)
    
    # Load pipeline data
    if not os.path.exists('data/pipeline_complete.csv'):
        print("❌ Pipeline data not found. Run unified pipeline first.")
        return
    
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    df = df.sort_values('block_date')
    
    # Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df)
    
    if not short_model or not long_model:
        print("❌ Could not load or rebuild models")
        return
    
    # Take last 60 days
    test_period = df.iloc[-60:].copy()
    
    # Generate signals
    signals = []
    for i, row in test_period.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
        
        # Print trade signals
        if signal['action'] == 'ENTER':
            direction_icon = "🟢" if signal['direction'] == 'LONG' else "🔴"
            print(f"{direction_icon} {signal['date']}: {signal['action']} {signal['direction']} "
                  f"@ ${row['eth_price']:.0f} (conf: {signal['adjusted_confidence']:.2f}, "
                  f"size: {signal['position_size']:.2f})")
    
    # Analyze results
    df_signals = pd.DataFrame(signals)
    
    print(f"\n📊 Test Results (60 days):")
    print(f"   Total days: {len(df_signals)}")
    print(f"   ENTER signals: {(df_signals['action'] == 'ENTER').sum()}")
    print(f"   LONG signals: {(df_signals['direction'] == 'LONG').sum()}")
    print(f"   SHORT signals: {(df_signals['direction'] == 'SHORT').sum()}")
    
    # Manual review questions
    if (df_signals['action'] == 'ENTER').sum() > 0:
        print(f"\n🔍 Manual Review Questions:")
        print(f"   1. Do LONGs occur only in R1/R2?")
        print(f"   2. Do SHORTs occur only in R3/R5?")
        print(f"   3. Are LONGs buying strength, not tops?")
        print(f"   4. Are SHORTs selling weakness/distribution?")
        print(f"   5. Are position sizes reasonable for regime?")
    
    # Regime distribution
    print(f"\n📈 Regime Distribution:")
    regime_dist = test_period['regime_code'].value_counts()
    for regime, count in regime_dist.items():
        print(f"   {regime}: {count} days ({count/len(test_period)*100:.1f}%)")
    
    return df_signals

# ========== MANUAL SIGNAL REVIEW ==========
def manual_signal_review(num_days=30):
    """
    Manual review of signals with guided questions
    """
    print("\n" + "="*70)
    print(f"MANUAL SIGNAL REVIEW ({num_days} days)")
    print("="*70)
    
    # Load pipeline data
    if not os.path.exists('data/pipeline_complete.csv'):
        print("❌ Pipeline data not found. Run unified pipeline first.")
        return
    
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    df = df.sort_values('block_date')
    
    # Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df)
    
    if not short_model or not long_model:
        print("❌ Could not load or rebuild models")
        return
    
    # Generate signals
    recent_data = df.iloc[-num_days:].copy()
    signals = []
    
    for i, row in recent_data.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
    
    # Review template
    print("\n📝 Review Template (for each ENTER signal):")
    print("-" * 40)
    
    for signal in signals:
        if signal['action'] == 'ENTER':
            print(f"\n📅 {signal['date']} - {signal['direction']} in {signal['regime']}")
            print(f"   Confidence: {signal['adjusted_confidence']:.2f}")
            print(f"   Position Size: {signal['position_size']:.2f}")
            print(f"   Reasons: {', '.join(signal['reasons'])}")
            
            # Find the row data
            row = df[df['block_date'] == pd.Timestamp(signal['date'])]
            if not row.empty:
                row = row.iloc[0]
                print(f"   ETH Price: ${row['eth_price']:.0f}")
                print(f"   BTC Ret Lag1: {row.get('btc_ret_lag1', 0):.3f}")
                print(f"   ETH Ret Lag1: {row.get('eth_ret_lag1', 0):.3f}")
            
            # Review questions
            if signal['direction'] == 'LONG':
                print("   Questions:")
                print("   1. Is price breaking structure upward?")
                print("   2. Is BTC aligned or neutral?")
                print("   3. Are we buying strength, not tops?")
            else:
                print("   Questions:")
                print("   1. Is this distribution or panic?")
                print("   2. Is liquidity present?")
                print("   3. Is this early weakness (R3) or real distribution (R5)?")
            
            print("-" * 40)
    
    return signals
def verify_vol_ratio_integration():
    """Verify vol_ratio is properly integrated"""
    print("\n" + "="*70)
    print("VOL_RATIO INTEGRATION VERIFICATION")
    print("="*70)
    
    # Load data
    df_whales, df_market, df_btc, df_eth = load_data_from_files()
    df_features = engineer_features(df_whales, df_market, df_btc, df_eth)
    
    print("\n📊 Vol Ratio Statistics:")
    if 'vol_ratio' in df_features.columns:
        print(f"✅ vol_ratio column exists")
        print(f"   Mean: {df_features['vol_ratio'].mean():.3f}")
        print(f"   Min: {df_features['vol_ratio'].min():.3f}")
        print(f"   Max: {df_features['vol_ratio'].max():.3f}")
        print(f"   < 1.0 (compression): {(df_features['vol_ratio'] < 1.0).sum()} days")
        print(f"   > 1.0 (expansion): {(df_features['vol_ratio'] > 1.0).sum()} days")
        
        # Check recent values
        recent = df_features.tail(10)
        print(f"\n📅 Recent vol_ratio values:")
        for i, row in recent.iterrows():
            date = row['block_date'].date()
            vol_ratio = row['vol_ratio']
            eth_vol7 = row.get('eth_vol7', 0)
            eth_vol30 = row.get('eth_vol30', 1)
            status = "📉 COMPRESSION" if vol_ratio < 1.0 else "📈 EXPANSION"
            print(f"   {date}: {vol_ratio:.3f} ({status}) | 7d:{eth_vol7:.4f} / 30d:{eth_vol30:.4f}")
    else:
        print("❌ vol_ratio column missing!")
    
    # Check if models will use vol_ratio
    print(f"\n🔍 Feature List Check:")
    print(f"   vol_ratio in LONG_FEATURES: {'vol_ratio' in LONG_FEATURES}")
    print(f"   vol_ratio in SHORT_FEATURES: {'vol_ratio' in SHORT_FEATURES}")
    
    # Build pipeline to check model rebuilding
    df_pipeline = build_pipeline_complete(df_features)
    short_model, long_model = rebuild_models_if_needed(df_pipeline)
    
    if short_model and hasattr(short_model, 'feature_names_'):
        print(f"\n✅ SHORT model features:")
        print(f"   {short_model.feature_names_}")
        print(f"   Uses vol_ratio: {'vol_ratio' in short_model.feature_names_}")
    
    if long_model and hasattr(long_model, 'feature_names_'):
        print(f"\n✅ LONG model features:")
        print(f"   {long_model.feature_names_}")
        print(f"   Uses vol_ratio: {'vol_ratio' in long_model.feature_names_}")

# ========== MAIN EXECUTION ==========
if __name__ == "__main__":
    print("\n" + "="*70)
    print("ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT")
    print("="*70)
    print("PHASE: EXECUTION HARDENING (Features FROZEN)")
    print(f"{'='*70}")
    
    # Check if data files exist
    required_files = [
        'data/whale_ml_ready.csv',
        'data/market_intent_ml_ready.csv', 
        'data/price_cache/btc.csv',
        'data/price_cache/eth.csv'
    ]
    
    missing_files = [f for f in required_files if not os.path.exists(f)]
    
    if missing_files:
        print(f"\n⚠️  Missing data files:")
        for f in missing_files:
            print(f"   - {f}")
        print(f"\nPlease run data_loader.py first to fetch data")
        print(f"Or place the required CSV files in the data directory")
        exit(1)
    
        # Ask user what to do
        print("\n📋 Available Options:")
        print("   1. Run unified pipeline (train models + generate signal)")
        print("   2. 60-day paper trade test (unified logic)")
        print("   3. Manual signal review (30 days)")
        print("   4. Inspect signals (60 days)")
        print("   5. Load and check data only")
        print("   6. Extended LONG inspection (2020-2021 bull cycle)")  # NEW OPTION
    
    choice = input("\nSelect option (1-6): ").strip()
    
    if choice == '1':
        df_pipeline, short_model, long_model = run_unified_pipeline()
        
        if df_pipeline is not None:
            print("\n✅ Unified pipeline complete")
            print("\n📋 Next steps:")
            print("   1. Review validation/signal_inspection_unified.csv")
            print("   2. Run option 2 for paper trade test")
            print("   3. Run option 3 for manual review")
    
    elif choice == '2':
        signals = run_unified_paper_test()
        
        print("\n📋 Review questions answered:")
        print("   ✅ LONGs only in R1/R2?")
        print("   ✅ SHORTs only in R3/R5?")
        print("   ✅ Position sizing consistent?")
        print("   ✅ Confidence ranges reasonable?")
    
    elif choice == '3':
        num_days = input("How many days to review? (default: 30): ").strip()
        try:
            num_days = int(num_days) if num_days else 30
        except:
            num_days = 30
        
        signals = manual_signal_review(num_days)
    
    elif choice == '4':
        # Load pipeline data
        if not os.path.exists('data/pipeline_complete.csv'):
            print("❌ Pipeline data not found. Run option 1 first.")
        else:
            df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
            short_model, long_model = rebuild_models_if_needed(df)
            
            if short_model and long_model:
                signals = inspect_signals_unified(df, short_model, long_model, 60)
    
    elif choice == '5':
        print("\n📂 Loading and checking data...")
        df_whales, df_market, df_btc, df_eth = load_data_from_files()
        
        print(f"\n✅ Data loaded successfully:")
        print(f"   Whale data: {len(df_whales)} rows")
        print(f"   Market data: {len(df_market)} rows")
        print(f"   BTC price: {len(df_btc)} rows")
        print(f"   ETH price: {len(df_eth)} rows")
    
    elif choice == '6':
        print("\n" + "="*70)
        print("EXTENDED LONG INSPECTION (2020-2021 BULL CYCLE)")
        print("="*70)
        
        # Load pipeline data
        if not os.path.exists('data/pipeline_complete.csv'):
            print("❌ Pipeline data not found. Run option 1 first.")
        else:
            df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
            df = df.sort_values('block_date')
            
            # Load models
            short_model_path = 'models/r5_short_final.pkl'
            long_model_path = 'models/R1_R2_LONG.pkl'
            
            if os.path.exists(short_model_path) and os.path.exists(long_model_path):
                short_model = joblib.load(short_model_path)
                long_model = joblib.load(long_model_path)
                
                # Ask for date range
                print("\n📅 Select inspection period:")
                print("   1. 2020-2021 bull cycle (2020-01-01 to 2022-01-01)")
                print("   2. Full available history")
                print("   3. Custom date range")
                
                period_choice = input("Select (1-3): ").strip()
                
                if period_choice == '1':
                    start_date = '2020-01-01'
                    end_date = '2022-01-01'
                elif period_choice == '2':
                    start_date = df['block_date'].min().strftime('%Y-%m-%d')
                    end_date = df['block_date'].max().strftime('%Y-%m-%d')
                elif period_choice == '3':
                    start_date = input("Start date (YYYY-MM-DD): ").strip()
                    end_date = input("End date (YYYY-MM-DD): ").strip()
                else:
                    start_date = '2020-01-01'
                    end_date = '2022-01-01'
                
                # Run inspection
                long_signals = inspect_long_signals_bull_cycle(
                    df, short_model, long_model, 
                    start_date=start_date, 
                    end_date=end_date
                )
                
                # Save results
                if long_signals:
                    df_results = pd.DataFrame(long_signals)
                    df_results.to_csv('validation/long_signals_bull_cycle.csv', index=False)
                    print(f"\n✅ Saved: validation/long_signals_bull_cycle.csv")
            else:
                print("❌ Models not found. Run option 1 first.")
    else:
        print("❌ Invalid option")
        
    print("\n" + "="*70)
    print("UNIFIED SIGNAL CONTRACT - SUMMARY")
    print("="*70)
    print("✅ Features: FROZEN")
    print("✅ Regimes: FROZEN")
    print("✅ Confidence: Unified (saturation + caps)")
    print("✅ Sizing: Unified (regime-aware caps)")
    print("✅ Signal Object: Consistent")
    print(f"{'='*70}")


ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT
PHASE: EXECUTION HARDENING (Features FROZEN)

ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT
✅ Features FROZEN
✅ Regimes FROZEN
✅ Funding: Veto-only from data loader
✅ Volatility: vol_ratio feature
📂 Loading data from saved files...
✅ Loaded whale_data: 3005 rows
✅ Loaded market_intent: 3005 rows
✅ Loaded btc_price: 4635 rows
✅ Loaded eth_price: 3805 rows
✅ Loaded funding_data: 3005 rows
   ✅ Full coverage: 2017-10-16 to 2026-01-06

✅ Essential data loaded successfully
🔧 Engineering features...
✅ Merged funding data: 3005 rows
   Funding coverage: 3005/3005 rows (100.0%)
✅ Created vol_ratio feature
⚠️  Creating missing SHORT feature: exchange_volume_zscore
✅ Features engineered: 56 columns, 3005 rows
   LONG features available: 16/16
   SHORT features available: 10/10

BUILDING COMPLETE PIPELINE DATASET
🎯 Creating two-tier targets...

📊 Target Distribution (Two-Tier SHORT):
  DOWN :  497 ( 16.5%)
  FLAT : 1915 ( 63.7%)
  UP   :  5